# ADME ACZ Silver Layer

Use this notebook to transform OSDU records from an Azure Data Manager for Energy Analytics Consumption Zone (ACZ) into Silver Layer Delta tables.

Before you run it, confirm that the customer has:

- A running Azure Data Manager for Energy instance.
- A configured Analytics Consumption Zone.
- A Microsoft Fabric lakehouse with access to the ACZ bronze Delta table.

For the full overview, prerequisites, configuration reference, and operating guidance, see `README.md`.


## Architecture

The notebook executes the Silver Layer transformation pipeline in Microsoft Fabric:

1. Read OSDU records from the ACZ bronze Delta table.
2. Resolve OSDU kind schemas from the ADME schema service.
3. Infer Spark types and flatten scalar and object fields.
4. Create parent and child Delta tables, or create one reassembled table per kind.
5. Write Silver Layer Delta tables for downstream analytics, reporting, and data engineering workloads.

The README owns the durable architecture and operating guidance. This notebook keeps the executable steps close to the code cells that run them.


## Spark runtime configuration

Prepare Spark before loading pipeline logic. The next cell reuses the active Fabric Spark session when available, creates one outside Fabric when needed, and applies safe defaults for Delta writes and decomposition workloads.


In [ ]:
import logging
import os

from pyspark.sql import SparkSession

# Configure logging at the beginning
logging.basicConfig(
    level=logging.WARN,
    format='%(asctime)s - %(levelname)s - %(filename)s:%(lineno)d - %(message)s'
)

# Fabric provides the active SparkSession in this notebook.
if "spark" not in globals() or spark is None:
    raise RuntimeError(
        "This notebook expects the Fabric-provided spark session. Attach the notebook to a Fabric runtime and rerun."
    )


def _env_or_default(name: str, default: str) -> str:
    val = os.environ.get(name)
    return val if val not in (None, "") else default


SPARK_CONFIG_DEFAULTS = {
    # Core SQL settings
    "spark.sql.session.timeZone": _env_or_default("ADME_SPARK_TIMEZONE", "UTC"),
    "spark.sql.adaptive.enabled": _env_or_default("ADME_SPARK_ADAPTIVE", "true"),
    "spark.sql.adaptive.coalescePartitions.enabled": _env_or_default("ADME_SPARK_COALESCE", "true"),
    "spark.sql.shuffle.partitions": _env_or_default("ADME_SPARK_SHUFFLE_PARTITIONS", "200"),
    "spark.sql.execution.arrow.pyspark.enabled": _env_or_default("ADME_SPARK_ARROW", "true"),
    # Delta Lake write optimizations
    "spark.microsoft.delta.optimizeWrite.enabled": _env_or_default("ADME_DELTA_OPTIMIZE_WRITE", "true"),
    "spark.microsoft.delta.optimizeWrite.binSize": _env_or_default("ADME_DELTA_BIN_SIZE", "1073741824"),  # 1GB
    "spark.databricks.delta.autoCompact.enabled": _env_or_default("ADME_DELTA_AUTO_COMPACT", "true"),
    # Fabric DirectLake compatibility (V-Order)
    "spark.sql.parquet.vorder.enabled": _env_or_default("ADME_PARQUET_VORDER", "true"),
    # File size control for better performance
    "spark.sql.files.maxPartitionBytes": _env_or_default("ADME_MAX_PARTITION_BYTES", "536870912"),  # 512MB
}

# Optional schema evolution style writes.
if _env_or_default("ADME_SPARK_AUTO_MERGE", "false").lower() == "true":
    SPARK_CONFIG_DEFAULTS["spark.databricks.delta.schema.autoMerge.enabled"] = "true"

for k, v in SPARK_CONFIG_DEFAULTS.items():
    spark.conf.set(k, v)

print("Spark session ready")
print(f"  appName: {spark.sparkContext.appName}")
print("Effective Spark config:")
for k in sorted(SPARK_CONFIG_DEFAULTS):
    try:
        print(f"  {k} = {spark.conf.get(k)}")
    except Exception:
        print(f"  {k} = <unavailable>")

## Configuration

Start here when moving the notebook to a new customer, tenant, workspace, or lakehouse. The next cell is split into two parts:

- **Customer settings**: values most runs need you to review or edit.
- **Advanced defaults**: operational controls that should usually stay unchanged until you need incremental refresh, schema-cache tuning, metadata-table changes, or retry/performance tuning.

For a first run, edit the customer settings, keep `RUN_PROFILE = "interactive"`, run the Setup checklist and Smoke test sections, then switch to `RUN_PROFILE = "dry_run"`. Use `RUN_PROFILE = "full"` only after the dry run shows the intended tables.


In [ ]:
import hashlib
import json
import os

# ===================================================================
# CUSTOMER SETTINGS - start here
# ===================================================================
# For most customer onboarding runs, edit only this section. Leave the
# advanced defaults below unchanged unless you need a specific behavior.

# Fabric target. Leave blank to use the attached Fabric lakehouse context.
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
BRONZE_TABLE = "osducatalog"

# ADME schema service. Both values are required for schema lookup.
ADME_ENDPOINT = ""
ADME_DATA_PARTITION_ID = ""

# ADME authentication. Use SP for direct notebook runs; use DC only for interactive validation.
ADME_AUTH_METHOD = "SP"  # "SP" | "DC"
ADME_TENANT_ID = ""
ADME_SP_CLIENT_ID = ""
ADME_SP_SECRET_KV_NAME = ""
ADME_SP_SECRET_NAME = ""

# Run stage. Start with interactive, then dry_run, then full.
RUN_PROFILE = "interactive"  # "interactive" | "dry_run" | "full"

# OSDU kinds to process. Start narrow; use wildcards only after a clean dry run.
KINDS = [
    "osdu:wks:work-product-component--WellLog:1.4.0",
    "osdu:wks:work-product-component--WellboreTrajectory:1.3.0",
]
LIMIT = 0                    # max records per kind; 0 = no limit
KIND_LIMITS = {}             # optional per-kind limits, e.g. {"WellLog": 100}

# Output shape and table safety.
OUTPUT_MODE = "normalized"  # "normalized" = parent+children | "wide" = one table per kind
TABLE_PREFIX = ""           # optional prefix for test or tenant-isolated outputs
ALLOW_OVERWRITE = False      # set True only when replacing existing output tables is intended

# ===================================================================
# ADVANCED DEFAULTS - keep unchanged unless needed
# ===================================================================

NOTEBOOK_VERSION = "0.2.0"

# Source record selection controls
INCLUDE_INACTIVE_RECORDS = False  # False = transform only rows where isActive == true

# Incremental/upsert controls
WRITE_MODE = "full_refresh"  # "full_refresh" | "upsert"
MERGE_KEY_COLUMNS = ["id", "version"]  # preserve multiple OSDU record versions per id
INCREMENTAL_WATERMARK_COLUMN = ""       # optional bronze column for source-change filtering in upsert mode
INCREMENTAL_WATERMARK_MODE = "auto"     # "off" | "auto" | "required"
INCREMENTAL_STATE_TABLE = "silver_incremental_state"

# Schema and table-shape controls
VERSION_STRATEGY = "versioned_tables"     # "merge" = union schema versions | "versioned_tables" = suffix tables by version
MISSING_SCHEMA_MODE = "skip"   # "skip" | "infer" | "fail"
CREATE_EMPTY_CHILD_TABLES = True
DROP_WKT = False                # True = drop WKT geometry columns

# Repeatability and metadata helpers
PERSIST_SCHEMA_CACHE = True
SCHEMA_CACHE_TABLE = "silver_schema_cache"
RUN_MANIFEST_TABLE = "silver_run_manifest"
WRITE_OUTPUT_DOCS = True
OUTPUT_DOCS_MODE = "summary"  # "summary" | "full" | "off"
OUTPUT_DOCS_TABLE = "silver_output_documentation"

# Performance controls
CACHE_BRONZE = True
BRONZE_CACHE_STORAGE_LEVEL = "MEMORY_AND_DISK"
PREFLIGHT_KIND_COUNTS = True
BATCH_METADATA_WRITES = True
METADATA_FLUSH_INTERVAL = 100
SCHEMA_PREFLIGHT = True

# ADME schema service retry controls
ADME_SCHEMA_TIMEOUT_SECONDS = 30
ADME_SCHEMA_RETRY_TOTAL = 3
ADME_SCHEMA_RETRY_BACKOFF_SECONDS = 1.0
ADME_SCHEMA_RETRY_STATUS_CODES = [408, 429, 500, 502, 503, 504]

# ════════════════════════════════════════════════════════════════════
# FABRIC RUNTIME RESOLUTION
# ════════════════════════════════════════════════════════════════════


def _spark_conf_get_optional(key: str) -> str | None:
    try:
        try:
            val = spark.conf.get(key, None)
        except TypeError:
            val = spark.conf.get(key)
        if val is None:
            return None
        sval = str(val).strip()
        return sval if sval else None
    except Exception:
        return None


def _resolve_workspace_id(explicit_value: str | None = None) -> str:
    explicit = (explicit_value or "").strip()
    if explicit:
        return explicit

    val = _spark_conf_get_optional("trident.workspace.id")
    if val:
        print(f"  spark.conf[trident.workspace.id] = {val}")
        return val
    raise ValueError(
        "Fabric workspace id could not be resolved. Attach a lakehouse, set WORKSPACE_ID, or set ADME_WORKSPACE_ID."
    )


def _resolve_lakehouse_id(explicit_value: str | None = None) -> str:
    explicit = (explicit_value or "").strip()
    if explicit:
        return explicit

    candidate_keys = [
        "trident.lakehouse.id",
        "trident.defaultLakehouse.id",
        "trident.lakehouseID",
    ]
    for key in candidate_keys:
        val = _spark_conf_get_optional(key)
        if val:
            print(f"  spark.conf[{key}] = {val}")
            return val
        print(f"  spark.conf[{key}] unavailable in this runtime")
    raise ValueError(
        "Fabric lakehouse id could not be resolved. Attach a lakehouse, set LAKEHOUSE_ID, or set ADME_LAKEHOUSE_ID."
    )


def _normalize_output_mode(value: str | None) -> str:
    normalized = (value or "normalized").strip().lower().replace("-", "_")
    aliases = {
        "normalized": "normalized",
        "normalised": "normalized",
        "parent_child": "normalized",
        "parent_children": "normalized",
        "parent+children": "normalized",
        "wide": "wide",
        "flat": "wide",
        "reassembled": "wide",
        "reassemble": "wide",
    }
    if normalized not in aliases:
        raise ValueError("OUTPUT_MODE must be 'normalized' or 'wide'.")
    return aliases[normalized]


def _normalize_version_strategy(value: str | None) -> str:
    normalized = (value or "merge").strip().lower().replace("-", "_")
    if normalized not in {"merge", "versioned_tables"}:
        raise ValueError("VERSION_STRATEGY must be 'merge' or 'versioned_tables'.")
    return normalized


def _normalize_missing_schema_mode(value: str | None) -> str:
    normalized = (value or "skip").strip().lower().replace("-", "_")
    if normalized not in {"skip", "infer", "fail"}:
        raise ValueError("MISSING_SCHEMA_MODE must be 'skip', 'infer', or 'fail'.")
    return normalized


def _normalize_output_docs_mode(value: str | None) -> str:
    normalized = (value or "summary").strip().lower().replace("-", "_")
    if normalized not in {"off", "summary", "full"}:
        raise ValueError("OUTPUT_DOCS_MODE must be 'off', 'summary', or 'full'.")
    return normalized


def _normalize_write_mode(value: str | None, incremental_flag: bool) -> str:
    normalized = (value or "").strip().lower().replace("-", "_")
    if not normalized:
        return "upsert" if incremental_flag else "full_refresh"
    aliases = {
        "full": "full_refresh",
        "full_refresh": "full_refresh",
        "overwrite": "full_refresh",
        "upsert": "upsert",
        "incremental": "upsert",
        "merge": "upsert",
    }
    if normalized not in aliases:
        raise ValueError("WRITE_MODE must be 'full_refresh' or 'upsert'.")
    return aliases[normalized]


def _normalize_watermark_mode(value: str | None) -> str:
    normalized = (value or "auto").strip().lower().replace("-", "_")
    if normalized not in {"off", "auto", "required"}:
        raise ValueError("INCREMENTAL_WATERMARK_MODE must be 'off', 'auto', or 'required'.")
    return normalized


def _parse_retry_status_codes(value: str | list[int] | tuple[int, ...] | None) -> list[int]:
    raw_values = ADME_SCHEMA_RETRY_STATUS_CODES if value in (None, "") else value
    if isinstance(raw_values, str):
        if raw_values.strip().startswith("["):
            loaded = json.loads(raw_values)
            if not isinstance(loaded, list):
                raise ValueError("ADME_SCHEMA_RETRY_STATUS_CODES JSON must be a list.")
            raw_values = loaded
        else:
            raw_values = [part.strip() for part in raw_values.split(",") if part.strip()]
    parsed = sorted({int(code) for code in raw_values})
    if any(code < 100 or code > 599 for code in parsed):
        raise ValueError("ADME_SCHEMA_RETRY_STATUS_CODES must contain HTTP status codes.")
    return parsed


def _env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y"}


def _parse_kind_limits(value: str | dict | None) -> dict[str, int]:
    if value in (None, ""):
        return {}

    if isinstance(value, dict):
        raw_items = value.items()
    else:
        text = str(value).strip()
        if not text:
            return {}
        if text.startswith("{"):
            loaded = json.loads(text)
            if not isinstance(loaded, dict):
                raise ValueError("ADME_KIND_LIMITS JSON must be an object.")
            raw_items = loaded.items()
        else:
            pairs = [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]
            raw_items = []
            for pair in pairs:
                if "=" not in pair:
                    raise ValueError("ADME_KIND_LIMITS entries must use key=value format.")
                key, raw_limit = pair.split("=", 1)
                raw_items.append((key.strip(), raw_limit.strip()))

    parsed: dict[str, int] = {}
    for key, raw_limit in raw_items:
        clean_key = str(key).strip()
        if not clean_key:
            raise ValueError("KIND_LIMITS contains an empty kind key.")
        limit_value = int(raw_limit)
        if limit_value < 0:
            raise ValueError("KIND_LIMITS values must be greater than or equal to 0.")
        parsed[clean_key] = limit_value
    return parsed


def _parse_merge_key_columns(value: str | list[str] | None) -> list[str]:
    if value in (None, ""):
        raw_values = MERGE_KEY_COLUMNS
    elif isinstance(value, list):
        raw_values = value
    else:
        text = str(value).strip()
        if text.startswith("["):
            loaded = json.loads(text)
            if not isinstance(loaded, list):
                raise ValueError("ADME_MERGE_KEY_COLUMNS JSON must be a list.")
            raw_values = loaded
        else:
            raw_values = [part.strip() for part in text.split(",")]

    parsed = [str(col).strip() for col in raw_values if str(col).strip()]
    if not parsed:
        raise ValueError("MERGE_KEY_COLUMNS must contain at least one column.")
    return list(dict.fromkeys(parsed))


def _required_config_value(name: str, value: str | None) -> str:
    clean = (value or "").strip()
    if not clean:
        raise ValueError(f"{name} must be set in the explicit tenant configuration block or as an environment variable.")
    return clean


def _normalize_adme_endpoint(value: str | None) -> str:
    endpoint = _required_config_value("ADME_ENDPOINT", value).rstrip("/")
    if not endpoint.lower().startswith("https://"):
        raise ValueError("ADME_ENDPOINT must be an HTTPS URL, for example https://contoso.energy.azure.com.")
    return endpoint


def _normalize_adme_auth_method(value: str | None) -> str:
    normalized = (value or "SP").strip().upper()
    if normalized not in {"SP", "DC"}:
        raise ValueError("ADME_AUTH_METHOD must be 'SP' or 'DC'.")
    return normalized


print("=== Configuration ===")

# Resolve workspace/lakehouse (env var takes precedence over explicit config, then spark conf)
_ws_env = os.environ.get("ADME_WORKSPACE_ID")
print(f"  ADME_WORKSPACE_ID env = {_ws_env!r}")
workspace_id = _ws_env or _resolve_workspace_id(WORKSPACE_ID)

_lh_env = os.environ.get("ADME_LAKEHOUSE_ID")
print(f"  ADME_LAKEHOUSE_ID env = {_lh_env!r}")
lakehouse_id = _lh_env or _resolve_lakehouse_id(LAKEHOUSE_ID)

bronze_table = os.environ.get("ADME_BRONZE_TABLE", BRONZE_TABLE)

adme_endpoint = _normalize_adme_endpoint(os.environ.get("ADME_ENDPOINT") or ADME_ENDPOINT)
adme_data_partition_id = _required_config_value("ADME_DATA_PARTITION_ID", os.environ.get("ADME_DATA_PARTITION_ID") or ADME_DATA_PARTITION_ID)
adme_auth_method = _normalize_adme_auth_method(os.environ.get("ADME_AUTH_METHOD") or ADME_AUTH_METHOD)
adme_tenant_id = _required_config_value("ADME_TENANT_ID", os.environ.get("ADME_TENANT_ID") or ADME_TENANT_ID)
adme_sp_client_id = (os.environ.get("ADME_SP_CLIENT_ID") or ADME_SP_CLIENT_ID).strip()
adme_sp_secret_kv_name = (os.environ.get("ADME_SP_SECRET_KV_NAME") or ADME_SP_SECRET_KV_NAME).strip()
adme_sp_secret_name = (os.environ.get("ADME_SP_SECRET_NAME") or ADME_SP_SECRET_NAME).strip()
if adme_auth_method == "SP":
    adme_sp_client_id = _required_config_value("ADME_SP_CLIENT_ID", adme_sp_client_id)
    adme_sp_secret_kv_name = _required_config_value("ADME_SP_SECRET_KV_NAME", adme_sp_secret_kv_name)
    adme_sp_secret_name = _required_config_value("ADME_SP_SECRET_NAME", adme_sp_secret_name)

print(f"\n  workspace_id = {workspace_id}")
print(f"  lakehouse_id = {lakehouse_id}")
print(f"  bronze_table = {bronze_table}")
print(f"  adme_endpoint = {adme_endpoint}")
print(f"  adme_data_partition_id = {adme_data_partition_id}")
print(f"  adme_auth_method = {adme_auth_method}")
print(f"  adme_tenant_id = {adme_tenant_id}")
if adme_auth_method == "SP":
    print(f"  adme_sp_client_id = {adme_sp_client_id}")
    print(f"  adme_sp_secret_kv_name = {adme_sp_secret_kv_name}")
    print(f"  adme_sp_secret_name = {adme_sp_secret_name}")

# ════════════════════════════════════════════════════════════════════
# DERIVED CONFIGURATION
# ════════════════════════════════════════════════════════════════════

# OSDU kind selectors to process (env var override or user control)
_env_kinds = os.environ.get("ADME_KINDS")
if _env_kinds:
    kind_selectors = [k.strip() for k in _env_kinds.split(",") if k.strip()]
else:
    kind_selectors = [str(k).strip() for k in KINDS if str(k).strip()]

# Resolved after helper cells are loaded. Exact selectors remain unchanged.
kinds = kind_selectors

# Run profile (env var override or user control)
run_profile = os.environ.get("ADME_RUN_PROFILE") or RUN_PROFILE
run_profile = run_profile.strip().lower()
if run_profile not in {"interactive", "dry_run", "full"}:
    run_profile = "interactive"

# Pipeline behavior (env var override or user control)
_write_mode_env = os.environ.get("ADME_WRITE_MODE")
_legacy_incremental_env = _env_bool("ADME_INCREMENTAL", False)
if _write_mode_env:
    write_mode = _normalize_write_mode(_write_mode_env, False)
elif _legacy_incremental_env:
    # Backward-compatible alias for older scheduled runs.
    write_mode = _normalize_write_mode("", True)
else:
    write_mode = _normalize_write_mode(WRITE_MODE, False)
incremental = write_mode == "upsert"
merge_key_columns = _parse_merge_key_columns(os.environ.get("ADME_MERGE_KEY_COLUMNS") or MERGE_KEY_COLUMNS)
watermark_column = (os.environ.get("ADME_INCREMENTAL_WATERMARK_COLUMN") or INCREMENTAL_WATERMARK_COLUMN).strip()
watermark_mode = _normalize_watermark_mode(os.environ.get("ADME_INCREMENTAL_WATERMARK_MODE") or INCREMENTAL_WATERMARK_MODE)
incremental_state_table = os.environ.get("ADME_INCREMENTAL_STATE_TABLE", INCREMENTAL_STATE_TABLE)
if watermark_mode == "required" and not watermark_column:
    raise ValueError("INCREMENTAL_WATERMARK_COLUMN must be set when INCREMENTAL_WATERMARK_MODE = 'required'.")
allow_overwrite = _env_bool("ADME_ALLOW_OVERWRITE", ALLOW_OVERWRITE)
limit_str = os.environ.get("ADME_LIMIT")
limit = int(limit_str) if limit_str else (LIMIT if LIMIT else None)
if limit == 0:
    limit = None
kind_limits = _parse_kind_limits(os.environ.get("ADME_KIND_LIMITS") or KIND_LIMITS)
include_inactive_records = _env_bool("ADME_INCLUDE_INACTIVE_RECORDS", INCLUDE_INACTIVE_RECORDS)

drop_wkt = _env_bool("ADME_DROP_WKT", DROP_WKT)

_output_mode_env = os.environ.get("ADME_OUTPUT_MODE")
if _output_mode_env:
    output_mode = _normalize_output_mode(_output_mode_env)
elif os.environ.get("ADME_REASSEMBLE"):
    # Backward-compatible alias for older scheduled runs.
    output_mode = "wide" if os.environ.get("ADME_REASSEMBLE", "").lower() != "false" else "normalized"
else:
    output_mode = _normalize_output_mode(OUTPUT_MODE)
reassemble = output_mode == "wide"
version_strategy = _normalize_version_strategy(os.environ.get("ADME_VERSION_STRATEGY", VERSION_STRATEGY))
missing_schema_mode = _normalize_missing_schema_mode(os.environ.get("ADME_MISSING_SCHEMA_MODE", MISSING_SCHEMA_MODE))
create_empty_child_tables = _env_bool("ADME_CREATE_EMPTY_CHILD_TABLES", CREATE_EMPTY_CHILD_TABLES)

table_prefix = os.environ.get("ADME_TABLE_PREFIX", TABLE_PREFIX)
persist_schema_cache = _env_bool("ADME_PERSIST_SCHEMA_CACHE", PERSIST_SCHEMA_CACHE)
schema_cache_table = os.environ.get("ADME_SCHEMA_CACHE_TABLE", SCHEMA_CACHE_TABLE)
run_manifest_table = os.environ.get("ADME_RUN_MANIFEST_TABLE", RUN_MANIFEST_TABLE)
output_docs_mode = _normalize_output_docs_mode(os.environ.get("ADME_OUTPUT_DOCS_MODE", OUTPUT_DOCS_MODE))
write_output_docs = _env_bool("ADME_WRITE_OUTPUT_DOCS", WRITE_OUTPUT_DOCS) and output_docs_mode != "off"
output_docs_table = os.environ.get("ADME_OUTPUT_DOCS_TABLE", OUTPUT_DOCS_TABLE)
cache_bronze = _env_bool("ADME_CACHE_BRONZE", CACHE_BRONZE)
bronze_cache_storage_level = os.environ.get("ADME_BRONZE_CACHE_STORAGE_LEVEL", BRONZE_CACHE_STORAGE_LEVEL)
preflight_kind_counts = _env_bool("ADME_PREFLIGHT_KIND_COUNTS", PREFLIGHT_KIND_COUNTS)
batch_metadata_writes = _env_bool("ADME_BATCH_METADATA_WRITES", BATCH_METADATA_WRITES)
metadata_flush_interval = int(os.environ.get("ADME_METADATA_FLUSH_INTERVAL", METADATA_FLUSH_INTERVAL))
schema_preflight = _env_bool("ADME_SCHEMA_PREFLIGHT", SCHEMA_PREFLIGHT)
adme_schema_timeout_seconds = int(os.environ.get("ADME_SCHEMA_TIMEOUT_SECONDS", ADME_SCHEMA_TIMEOUT_SECONDS))
adme_schema_retry_total = int(os.environ.get("ADME_SCHEMA_RETRY_TOTAL", ADME_SCHEMA_RETRY_TOTAL))
adme_schema_retry_backoff_seconds = float(os.environ.get("ADME_SCHEMA_RETRY_BACKOFF_SECONDS", ADME_SCHEMA_RETRY_BACKOFF_SECONDS))
adme_schema_retry_status_codes = _parse_retry_status_codes(os.environ.get("ADME_SCHEMA_RETRY_STATUS_CODES") or ADME_SCHEMA_RETRY_STATUS_CODES)
schema_cache_writes_enabled = persist_schema_cache and run_profile == "full"

config_snapshot = {
    "notebook_version": NOTEBOOK_VERSION,
    "workspace_id": workspace_id,
    "lakehouse_id": lakehouse_id,
    "bronze_table": bronze_table,
    "adme_endpoint": adme_endpoint,
    "adme_data_partition_id": adme_data_partition_id,
    "adme_auth_method": adme_auth_method,
    "adme_tenant_id": adme_tenant_id,
    "adme_sp_client_id": adme_sp_client_id if adme_auth_method == "SP" else "",
    "adme_sp_secret_kv_name": adme_sp_secret_kv_name if adme_auth_method == "SP" else "",
    "adme_sp_secret_name": adme_sp_secret_name if adme_auth_method == "SP" else "",
    "kind_selectors": kind_selectors,
    "run_profile": run_profile,
    "write_mode": write_mode,
    "incremental": incremental,
    "merge_key_columns": merge_key_columns,
    "watermark_column": watermark_column,
    "watermark_mode": watermark_mode,
    "incremental_state_table": incremental_state_table,
    "allow_overwrite": allow_overwrite,
    "limit": limit,
    "kind_limits": kind_limits,
    "include_inactive_records": include_inactive_records,
    "output_mode": output_mode,
    "version_strategy": version_strategy,
    "missing_schema_mode": missing_schema_mode,
    "create_empty_child_tables": create_empty_child_tables,
    "drop_wkt": drop_wkt,
    "table_prefix": table_prefix,
    "persist_schema_cache": persist_schema_cache,
    "schema_cache_table": schema_cache_table,
    "run_manifest_table": run_manifest_table,
    "write_output_docs": write_output_docs,
    "output_docs_mode": output_docs_mode,
    "output_docs_table": output_docs_table,
    "cache_bronze": cache_bronze,
    "bronze_cache_storage_level": bronze_cache_storage_level,
    "preflight_kind_counts": preflight_kind_counts,
    "batch_metadata_writes": batch_metadata_writes,
    "metadata_flush_interval": metadata_flush_interval,
    "schema_preflight": schema_preflight,
    "adme_schema_timeout_seconds": adme_schema_timeout_seconds,
    "adme_schema_retry_total": adme_schema_retry_total,
    "adme_schema_retry_backoff_seconds": adme_schema_retry_backoff_seconds,
    "adme_schema_retry_status_codes": adme_schema_retry_status_codes,
}
config_hash = hashlib.sha256(json.dumps(config_snapshot, sort_keys=True).encode("utf-8")).hexdigest()[:12]

# Schema source (ADME schema service with optional persisted cache)
schema_source_mode = "adme-cache" if persist_schema_cache else "adme"

# Print effective configuration
print("\n=== Effective Configuration ===")
print(f"  Notebook version: {NOTEBOOK_VERSION}")
print(f"  Config hash: {config_hash}")
print(f"  Run profile: {run_profile}")
print(f"  Schema source: {schema_source_mode}")
print(f"  ADME endpoint: {adme_endpoint}")
print(f"  ADME data partition id: {adme_data_partition_id}")
print(f"  ADME auth method: {adme_auth_method}")
print(f"  Kind selectors: {len(kind_selectors)}")
if kind_selectors:
    preview = kind_selectors[:3]
    print(f"    {preview}")
    if len(kind_selectors) > len(preview):
        print(f"    ... and {len(kind_selectors) - len(preview)} more")
print("  Wildcard selectors resolve after helper cells are loaded")
print(f"  Write mode: {write_mode}")
print(f"  Upsert mode enabled: {incremental}")
print(f"  Merge key columns: {merge_key_columns}")
print(f"  Incremental watermark column: {watermark_column or 'none'}")
print(f"  Incremental watermark mode: {watermark_mode}")
print(f"  Incremental state table: {incremental_state_table}")
print(f"  Allow overwrite: {allow_overwrite}")
print(f"  Record limit: {limit or 'none'}")
print(f"  Per-kind limits: {kind_limits or 'none'}")
print(f"  Include inactive records: {include_inactive_records}")
print(f"  Output mode: {output_mode}")
print(f"  Version strategy: {version_strategy}")
print(f"  Missing schema mode: {missing_schema_mode}")
print(f"  Create empty child tables: {create_empty_child_tables}")
print(f"  Drop WKT: {drop_wkt}")
print(f"  Table prefix: {table_prefix!r}")
print(f"  Persist schema cache: {persist_schema_cache}")
print(f"  Schema cache writes enabled: {schema_cache_writes_enabled}")
print(f"  Schema cache table: {schema_cache_table}")
print(f"  Run manifest table: {run_manifest_table}")
print(f"  Write output docs: {write_output_docs}")
print(f"  Output docs mode: {output_docs_mode}")
print(f"  Output docs table: {output_docs_table}")
print(f"  Cache bronze: {cache_bronze} ({bronze_cache_storage_level})")
print(f"  Preflight kind counts: {preflight_kind_counts}")
print(f"  Batch metadata writes: {batch_metadata_writes} every {metadata_flush_interval} result(s)")
print(f"  Schema preflight: {schema_preflight}")
print(f"  ADME schema timeout: {adme_schema_timeout_seconds}s")
print(f"  ADME schema retries: {adme_schema_retry_total} total, backoff {adme_schema_retry_backoff_seconds}s, statuses {adme_schema_retry_status_codes}")
print("=" * 32)


## Pipeline constants

Define shared endpoints, storage scopes, run metadata schema, and per-kind result types used by the pipeline.


In [ ]:
import traceback
import uuid
from dataclasses import dataclass
from datetime import UTC, datetime

from pyspark.sql import types as T

# ── External service endpoints ───────────────────────────────────────

# ADME schema service endpoint path, appended to ADME_ENDPOINT from configuration.
ADME_SCHEMA_SERVICE_PATH = "/api/schema-service/v1/schema"

# Static ADME authentication constants. These are not tenant-specific onboarding values.
ADME_TOKEN_SCOPE = "https://management.core.windows.net/.default"
ADME_DEVICE_CODE_CLIENT_ID = "04b07795-8ddb-461a-bbee-02f9e1bf7b46"

# Azure Storage scope for Fabric authentication
FABRIC_STORAGE_SCOPE = "https://storage.azure.com/.default"

# ── Run-info schema ──────────────────────────────────────────────────

RUN_INFO_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("start_time", T.TimestampType(), False),
        T.StructField("end_time", T.TimestampType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("error_message", T.StringType(), True),
        T.StructField("duration_seconds", T.DoubleType(), True),
        T.StructField("error_type", T.StringType(), True),
        T.StructField("write_mode", T.StringType(), True),
        T.StructField("output_mode", T.StringType(), True),
        T.StructField("merge_key_columns", T.ArrayType(T.StringType()), True),
        T.StructField("schema_access_detail", T.StringType(), True),
        T.StructField("stage_timings_json", T.StringType(), True),
        T.StructField("watermark_column", T.StringType(), True),
        T.StructField("watermark_mode", T.StringType(), True),
    ]
)

SCHEMA_CACHE_SCHEMA = T.StructType(
    [
        T.StructField("kind", T.StringType(), False),
        T.StructField("source", T.StringType(), False),
        T.StructField("schema_json", T.StringType(), False),
        T.StructField("cached_at", T.TimestampType(), False),
    ]
)

RUN_MANIFEST_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("output_mode", T.StringType(), False),
        T.StructField("version_strategy", T.StringType(), True),
        T.StructField("kind_group", T.StringType(), True),
        T.StructField("schema_versions", T.ArrayType(T.StringType()), True),
        T.StructField("schema_mode", T.StringType(), True),
        T.StructField("notebook_version", T.StringType(), True),
        T.StructField("run_profile", T.StringType(), True),
        T.StructField("config_hash", T.StringType(), True),
        T.StructField("allow_overwrite", T.BooleanType(), True),
        T.StructField("table_prefix", T.StringType(), True),
        T.StructField("bronze_table", T.StringType(), True),
        T.StructField("parent_table", T.StringType(), True),
        T.StructField("child_tables", T.ArrayType(T.StringType()), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("error_message", T.StringType(), True),
        T.StructField("created_at", T.TimestampType(), False),
        T.StructField("write_mode", T.StringType(), True),
        T.StructField("merge_key_columns", T.ArrayType(T.StringType()), True),
        T.StructField("child_table_count", T.IntegerType(), True),
        T.StructField("output_docs_mode", T.StringType(), True),
        T.StructField("schema_cache_enabled", T.BooleanType(), True),
        T.StructField("cache_bronze", T.BooleanType(), True),
        T.StructField("watermark_column", T.StringType(), True),
        T.StructField("watermark_mode", T.StringType(), True),
        T.StructField("include_inactive_records", T.BooleanType(), True),
    ]
)

INCREMENTAL_STATE_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("watermark_column", T.StringType(), False),
        T.StructField("watermark_value", T.StringType(), True),
        T.StructField("watermark_data_type", T.StringType(), True),
        T.StructField("updated_at", T.TimestampType(), False),
    ]
)

OUTPUT_DOCS_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("output_mode", T.StringType(), False),
        T.StructField("table_name", T.StringType(), False),
        T.StructField("table_role", T.StringType(), False),
        T.StructField("source_column", T.StringType(), True),
        T.StructField("column_name", T.StringType(), False),
        T.StructField("data_type", T.StringType(), False),
        T.StructField("ordinal", T.IntegerType(), False),
        T.StructField("nullable", T.BooleanType(), True),
        T.StructField("notebook_version", T.StringType(), True),
        T.StructField("config_hash", T.StringType(), True),
        T.StructField("created_at", T.TimestampType(), False),
    ]
)


@dataclass
class KindResult:
    """Outcome of processing one kind."""

    kind: str
    status: str
    records_processed: int = 0
    records_failed: int = 0
    parent_table: str = ""
    child_tables: list[str] | None = None
    reassembled: bool = False
    validation_passed: bool = True
    error: str | None = None

## Helper functions

Load Fabric, OneLake, Delta write, upsert, and bronze-read helpers. These helpers use the attached lakehouse catalog first and fall back to OneLake paths when a catalog table is not available.


In [ ]:
import logging
import os
from datetime import datetime
from typing import Any

import requests
from pyspark import StorageLevel
from azure.identity import DefaultAzureCredential
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

logger = logging.getLogger(__name__)


def _requires_path_fallback(exc: Exception) -> bool:
    msg = str(exc)
    return (
        "No default context found" in msg
        or "partial namespaces" in msg
        or "attach a lakehouse" in msg.lower()
    )


def _table_path_uri(table_name: str) -> str:
    ws = globals().get("workspace_id")
    lh = globals().get("lakehouse_id")
    if not ws or not lh:
        raise ValueError(
            "workspace_id/lakehouse_id not resolved; run Configuration cell before pipeline execution."
        )
    if "_onelake_table_uri" not in globals():
        raise ValueError("_onelake_table_uri is unavailable; run helper cells before pipeline execution.")
    return _onelake_table_uri(ws, lh, table_name)


# Fabric-only helpers
def kind_to_table_name(kind: str) -> str:
    # osdu:wks:master-data--Well:1.2.0 -> well
    # osdu:wks:work-product-component--WellLog:1.4.0 -> welllog
    try:
        _, _, entity_ver = kind.split(":", 2)
        entity, _ = entity_ver.rsplit(":", 1)
    except ValueError:
        entity = kind
    entity_base = entity.split("--")[-1]
    return entity_base.replace("-", "_").replace(".", "_").lower()


def table_uri(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    # In Fabric with attached lakehouse, tables are resolved via catalog.
    # Without attached context we transparently fall back to OneLake path.
    return table_name


def _write_table(df: DataFrame, target: str, mode: str = "overwrite") -> None:
    writer = df.write.format("delta").mode(mode)
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    elif mode == "append":
        writer = writer.option("mergeSchema", "true")
    try:
        writer.saveAsTable(target)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise
        target_uri = _table_path_uri(target)
        logger.info("No default lakehouse context; writing by path: %s", target_uri)
        writer.save(target_uri)
    if "_TABLE_EXISTS_CACHE" in globals():
        _TABLE_EXISTS_CACHE[target] = True


def write_silver_table(df: DataFrame, target: str, mode: str = "overwrite") -> None:
    _write_table(df, target, mode=mode)



def _effective_merge_key_columns(merge_key_columns: list[str] | None = None) -> list[str]:
    columns = merge_key_columns or globals().get("merge_key_columns", ["id", "version"])
    parsed = [str(col).strip() for col in columns if str(col).strip()]
    if not parsed:
        raise ValueError("At least one merge key column is required.")
    return parsed


def _validate_merge_key_columns(df: DataFrame, merge_key_columns: list[str], context: str) -> None:
    missing = [col for col in merge_key_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{context} is missing merge key column(s): {missing}. Available columns: {df.columns}")


def _merge_condition(target_alias: str, source_alias: str, merge_key_columns: list[str]) -> str:
    return " AND ".join(f"{target_alias}.`{col}` = {source_alias}.`{col}`" for col in merge_key_columns)


def _assert_no_duplicate_merge_keys(df: DataFrame, merge_key_columns: list[str], context: str) -> None:
    duplicates = df.groupBy(*[F.col(col) for col in merge_key_columns]).count().filter(F.col("count") > 1).limit(1).collect()
    if duplicates:
        key_preview = {col: duplicates[0][col] for col in merge_key_columns}
        raise ValueError(
            f"{context} contains duplicate rows for merge key {merge_key_columns}: {key_preview}. "
            "Deduplicate the source or choose merge keys that uniquely identify each row."
        )



def _read_target_df_for_merge(spark: SparkSession, target: str, by_path: bool = False) -> DataFrame:
    if by_path:
        return spark.read.format("delta").load(_table_path_uri(target))
    return spark.table(target)


def _align_source_to_target_schema(source_df: DataFrame, target_df: DataFrame) -> DataFrame:
    """Add missing target columns to source as typed nulls before Delta updateAll/insertAll."""
    aligned = source_df
    source_cols = set(aligned.columns)
    for field in target_df.schema.fields:
        if field.name not in source_cols:
            aligned = aligned.withColumn(field.name, F.lit(None).cast(field.dataType))
    return aligned


def _execute_delta_merge(
    delta_table,
    target_df: DataFrame,
    source_df: DataFrame,
    target: str,
    merge_key_columns: list[str],
    by_path: bool,
) -> None:
    source_alias = "s"
    target_alias = "t"
    merge_df = _align_source_to_target_schema(source_df, target_df)
    _validate_merge_key_columns(merge_df, merge_key_columns, f"Aligned source DataFrame for {target}")
    cond = _merge_condition(target_alias, source_alias, merge_key_columns)
    try:
        (
            delta_table.alias(target_alias)
            .merge(merge_df.alias(source_alias), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    except Exception as exc:
        location = "OneLake path" if by_path else "catalog table"
        raise RuntimeError(f"Delta merge failed for existing {location} {target!r}; not overwriting target table.") from exc


def upsert_silver_table(df: DataFrame, target: str, merge_key_columns: list[str] | None = None) -> None:
    from delta.tables import DeltaTable

    merge_keys = _effective_merge_key_columns(merge_key_columns)
    _validate_merge_key_columns(df, merge_keys, f"Source DataFrame for {target}")
    _assert_no_duplicate_merge_keys(df, merge_keys, f"Source DataFrame for {target}")

    try:
        exists = spark.catalog.tableExists(target)
    except Exception as exc:
        if _requires_path_fallback(exc):
            exists = False
        else:
            raise

    if exists:
        dt = DeltaTable.forName(spark, target)
        target_df = _read_target_df_for_merge(spark, target, by_path=False)
        _execute_delta_merge(dt, target_df, df, target, merge_keys, by_path=False)
        return

    target_uri = _table_path_uri(target)
    try:
        path_exists = DeltaTable.isDeltaTable(spark, target_uri)
    except Exception as exc:
        raise RuntimeError(f"Could not determine whether Delta target {target!r} exists at {target_uri!r}; refusing to overwrite.") from exc

    if path_exists:
        dt_path = DeltaTable.forPath(spark, target_uri)
        target_df = _read_target_df_for_merge(spark, target, by_path=True)
        _execute_delta_merge(dt_path, target_df, df, target_uri, merge_keys, by_path=True)
        return

    logger.info("Creating new Delta table for upsert target %s", target)
    _write_table(df, target, mode="overwrite")


# Fabric storage helpers

_FABRIC_CREDENTIAL = DefaultAzureCredential()


def _fabric_storage_options() -> dict[str, str]:
    token = _FABRIC_CREDENTIAL.get_token(FABRIC_STORAGE_SCOPE).token
    return {
        "bearer_token": token,
        "use_fabric_endpoint": "true",
    }


def _onelake_table_uri(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    lakehouse_ref = lakehouse_id or os.environ.get("ADME_LAKEHOUSE_NAME", "osducatalog")
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_ref}/Tables/{table_name}"


def _include_inactive_records() -> bool:
    return bool(globals().get("include_inactive_records", False))


def active_record_filter_status(df: DataFrame, include_inactive_records: bool | None = None) -> tuple[bool, str]:
    include_inactive = _include_inactive_records() if include_inactive_records is None else bool(include_inactive_records)
    if include_inactive:
        return True, "inactive records are included by configuration"
    if "isActive" not in df.columns:
        return False, "Bronze table must contain an 'isActive' column when INCLUDE_INACTIVE_RECORDS is False."
    return True, "excluding rows where isActive is not true"


def apply_active_record_filter(df: DataFrame, include_inactive_records: bool | None = None) -> DataFrame:
    include_inactive = _include_inactive_records() if include_inactive_records is None else bool(include_inactive_records)
    passed, detail = active_record_filter_status(df, include_inactive)
    if not passed:
        raise ValueError(detail)
    if include_inactive:
        return df
    return df.filter(F.col("isActive") == F.lit(True))


def read_bronze_table_spark(
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str | None = None,
    apply_active_filter: bool = True,
) -> DataFrame:
    bronze_table = bronze_table or globals().get("bronze_table")
    if not bronze_table:
        raise ValueError("bronze_table is not configured")

    # Try attached lakehouse table first; fall back to OneLake path when context is missing.
    df = None
    try:
        if spark.catalog.tableExists(bronze_table):
            df = spark.table(bronze_table)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise
        logger.info("No default lakehouse context; resolving bronze via OneLake path")

    if df is None:
        bronze_uri = _onelake_table_uri(workspace_id, lakehouse_id, bronze_table)
        logger.info("Bronze table not attached; trying OneLake path: %s", bronze_uri)
        try:
            df = spark.read.format("delta").load(bronze_uri)
        except Exception as exc:
            raise ValueError(
                f"Bronze table '{bronze_table}' was not found in the attached catalog and OneLake load failed from '{bronze_uri}'. "
                "Attach a lakehouse or ensure ADME_WORKSPACE_ID/ADME_LAKEHOUSE_ID/ADME_BRONZE_TABLE point to a valid OneLake Delta table."
            ) from exc

    return apply_active_record_filter(df) if apply_active_filter else df


def read_bronze_kind_spark(
    kind: str,
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str | None = None,
    limit: int | None = None,
    bronze_df: DataFrame | None = None,
) -> DataFrame:
    df = bronze_df if bronze_df is not None else read_bronze_table_spark(spark, workspace_id, lakehouse_id, bronze_table=bronze_table)

    if "kind" in df.columns:
        df = df.filter(F.col("kind") == F.lit(kind))

    return df.limit(limit) if limit else df

def _storage_level_from_name(name: str):
    normalized = (name or "MEMORY_AND_DISK").strip().upper()
    return getattr(StorageLevel, normalized, StorageLevel.MEMORY_AND_DISK)


def prepare_bronze_df(
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    cache_enabled: bool = True,
    storage_level_name: str = "MEMORY_AND_DISK",
) -> tuple[DataFrame, bool]:
    df = read_bronze_table_spark(spark, workspace_id, lakehouse_id, bronze_table=bronze_table)
    if "kind" not in df.columns:
        raise ValueError(f"Bronze table '{bronze_table}' does not contain a 'kind' column.")
    if not cache_enabled:
        return df, False
    try:
        df = df.persist(_storage_level_from_name(storage_level_name))
        logger.info("Bronze table cached with storage level %s", storage_level_name)
        return df, True
    except Exception as exc:
        logger.warning("Could not cache bronze table; continuing uncached: %s", exc)
        return df, False



## Core decomposition and reassembly logic

Load the standalone Silver Layer transformation logic. This section resolves OSDU schemas, classifies columns, builds parent and child tables, and optionally reassembles child data into one wide table per kind.

Do not edit this section for documentation cleanup. If transformation behavior changes are needed, update and test the source implementation first, then sync this notebook copy.


In [ ]:
# ── Core standalone schema/decompose/reassemble ─────────────────────
# This cell contains the standalone Silver Layer transformation implementation:
#   decompose.py
#   reassemble.py
#   schema_registry.py  (parsing + queries only — no Spark/Delta loaders)
#   naming.py           (child_table_name only — kind_to_table_name lives in helpers cell)
#   types.py            (DecomposedKind only)
#
# Only standalone-specific change: SchemaRegistry is built from the ADME schema service
# (load_schema_doc) instead of the osdu_schemas Delta table.
# DO NOT EDIT decomposition logic here unless you are intentionally changing transformation behavior.

from __future__ import annotations

import json
import logging
from collections import deque
from dataclasses import dataclass
from functools import reduce
from time import perf_counter, time
from datetime import UTC, datetime
from typing import Any
from urllib.parse import quote

import requests
from msal import ConfidentialClientApplication, PublicClientApplication
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

logger = logging.getLogger(__name__)

# ── ADME schema service loader ───────────────────────────────────────


def parse_kind(kind: str) -> dict[str, str]:
    authority, source, entity_ver = kind.split(":", 2)
    entity, version = entity_ver.rsplit(":", 1)
    return {
        "authority": authority,
        "source": source,
        "entity": entity,
        "version": version,
        "entity_base": entity.split("--")[-1],
    }


def normalize_kind_ref(ref: str | None) -> str | None:
    if not ref:
        return None
    normalized = ref.replace("{{schema-authority}}", "osdu")
    normalized = normalized.replace("{{wksNameSpace}}", "wks")
    normalized = normalized.replace("{{wksVersion}}", "1.0.0")
    return normalized


_SCHEMA_DOC_CACHE: dict[tuple[str, str, str], tuple[dict[str, Any], str]] = {}
_SCHEMA_LOAD_ERRORS: dict[str, str] = {}
_ADME_ACCESS_TOKEN: str | None = None
_ADME_ACCESS_TOKEN_EXPIRES_ON: int = 0


def _persistent_schema_cache_enabled() -> bool:
    return bool(globals().get("persist_schema_cache", False))


def _schema_cache_table_name() -> str:
    return globals().get("schema_cache_table", "silver_schema_cache")


def _read_table_for_cache(table_name: str) -> DataFrame:
    try:
        return spark.table(table_name)
    except Exception as exc:
        if "_requires_path_fallback" not in globals() or not _requires_path_fallback(exc):
            raise
        return spark.read.format("delta").load(_table_path_uri(table_name))


def _load_schema_docs_from_persistent_cache(kinds: list[str]) -> dict[str, tuple[dict[str, Any], str]]:
    unique_kinds = list(dict.fromkeys(kind for kind in kinds if kind))
    if not unique_kinds or not _persistent_schema_cache_enabled() or "spark" not in globals() or "_table_exists" not in globals():
        return {}

    table_name = _schema_cache_table_name()
    if not _table_exists(spark, table_name):
        return {}

    source_prefix = _adme_schema_source_prefix()
    try:
        latest_by_kind = Window.partitionBy("kind").orderBy(F.col("cached_at").desc())
        rows = (
            _read_table_for_cache(table_name)
            .filter(F.col("kind").isin(unique_kinds))
            .filter(F.col("source").startswith(source_prefix))
            .withColumn("__schema_cache_rank", F.row_number().over(latest_by_kind))
            .filter(F.col("__schema_cache_rank") == F.lit(1))
            .select("kind", "schema_json", "source")
            .collect()
        )
    except Exception as exc:
        logger.warning("Could not batch-read persisted schema cache; falling back to ADME schema service: %s", exc)
        return {}

    cached: dict[str, tuple[dict[str, Any], str]] = {}
    for row in rows:
        kind = str(row["kind"])
        try:
            cached[kind] = (json.loads(row["schema_json"]), str(row["source"]))
        except (TypeError, ValueError, json.JSONDecodeError) as exc:
            logger.warning("Ignoring invalid persisted schema cache row for %s: %s", kind, exc)
    return cached


def _load_schema_doc_from_persistent_cache(kind: str) -> tuple[dict[str, Any], str] | None:
    return _load_schema_docs_from_persistent_cache([kind]).get(kind)


def _write_schema_docs_to_persistent_cache(schema_rows: list[tuple[str, dict[str, Any], str]]) -> None:
    if not schema_rows or not globals().get("schema_cache_writes_enabled", False) or "spark" not in globals() or "_write_table" not in globals():
        return

    table_name = _schema_cache_table_name()
    cached_at = datetime.now(UTC)
    rows = [(kind, source, json.dumps(schema_doc, sort_keys=True), cached_at) for kind, schema_doc, source in schema_rows]
    df = spark.createDataFrame(rows, schema=SCHEMA_CACHE_SCHEMA)
    mode = "append" if "_table_exists" in globals() and _table_exists(spark, table_name) else "overwrite"
    _write_table(df, table_name, mode=mode)


def _write_schema_doc_to_persistent_cache(kind: str, schema_doc: dict[str, Any], source: str) -> None:
    _write_schema_docs_to_persistent_cache([(kind, schema_doc, source)])


def _adme_schema_config() -> tuple[str, str, str]:
    endpoint = str(globals().get("adme_endpoint") or globals().get("ADME_ENDPOINT") or "").strip().rstrip("/")
    data_partition_id = str(globals().get("adme_data_partition_id") or globals().get("ADME_DATA_PARTITION_ID") or "").strip()
    token_scope = str(globals().get("ADME_TOKEN_SCOPE") or "").strip()
    missing = [
        name
        for name, value in {
            "ADME_ENDPOINT": endpoint,
            "ADME_DATA_PARTITION_ID": data_partition_id,
            "ADME_TOKEN_SCOPE": token_scope,
        }.items()
        if not value
    ]
    if missing:
        raise ValueError(f"ADME schema service configuration is incomplete: {', '.join(missing)}.")
    if not endpoint.lower().startswith("https://"):
        raise ValueError("ADME_ENDPOINT must be an HTTPS URL, for example https://contoso.energy.azure.com.")
    return endpoint, data_partition_id, token_scope


def _adme_auth_method() -> str:
    method = str(globals().get("adme_auth_method") or globals().get("ADME_AUTH_METHOD") or "SP").strip().upper()
    if method not in {"SP", "DC"}:
        raise ValueError("ADME_AUTH_METHOD must be 'SP' or 'DC'.")
    return method


def _adme_auth_value(name: str, required: bool = True) -> str:
    lower_name = name.lower()
    value = str(globals().get(lower_name) or globals().get(name) or "").strip()
    if required and not value:
        raise ValueError(f"{name} must be configured for ADME { _adme_auth_method() } authentication.")
    return value


def _adme_authority_url() -> str:
    tenant_id = _adme_auth_value("ADME_TENANT_ID")
    return f"https://login.microsoftonline.com/{tenant_id}"


def _adme_keyvault_url(kv_name_or_url: str) -> str:
    value = kv_name_or_url.strip()
    if value.lower().startswith("https://"):
        return value.rstrip("/") + "/"
    return f"https://{value}.vault.azure.net/"


def _notebookutils_credentials():
    nb_utils = globals().get("notebookutils")
    if nb_utils is not None and hasattr(nb_utils, "credentials"):
        return nb_utils.credentials
    try:
        import notebookutils as _notebookutils

        if hasattr(_notebookutils, "credentials"):
            return _notebookutils.credentials
    except Exception:
        pass
    try:
        from notebookutils import mssparkutils as _mssparkutils

        return _mssparkutils.credentials
    except Exception as exc:
        raise RuntimeError("NotebookUtils credentials are required to read the ADME service principal secret from Key Vault.") from exc


def _adme_service_principal_secret() -> str:
    kv_name = _adme_auth_value("ADME_SP_SECRET_KV_NAME")
    secret_name = _adme_auth_value("ADME_SP_SECRET_NAME")
    return _notebookutils_credentials().getSecret(_adme_keyvault_url(kv_name), secret_name)


def _msal_token_result(result: dict[str, Any], context: str) -> tuple[str, int]:
    if "access_token" not in result:
        error = result.get("error") or "unknown_error"
        description = result.get("error_description") or "No error description returned by Microsoft Entra ID."
        raise RuntimeError(f"{context} failed: {error}, {description}")
    expires_on = int(result.get("expires_on") or (time() + int(result.get("expires_in", 3600))))
    return result["access_token"], expires_on


def _acquire_adme_access_token() -> tuple[str, int]:
    _, _, token_scope = _adme_schema_config()
    authority = _adme_authority_url()
    method = _adme_auth_method()
    if method == "SP":
        app = ConfidentialClientApplication(
            client_id=_adme_auth_value("ADME_SP_CLIENT_ID"),
            client_credential=_adme_service_principal_secret(),
            authority=authority,
        )
        return _msal_token_result(app.acquire_token_for_client(scopes=[token_scope]), "ADME service principal authentication")

    app = PublicClientApplication(
        client_id=_adme_auth_value("ADME_DEVICE_CODE_CLIENT_ID"),
        authority=authority,
    )
    flow = app.initiate_device_flow(scopes=[token_scope])
    if "message" not in flow:
        raise RuntimeError("ADME device code authentication failed to start: no device-code message returned.")
    print(flow["message"])
    return _msal_token_result(app.acquire_token_by_device_flow(flow), "ADME device code authentication")


def get_adme_access_token() -> str:
    global _ADME_ACCESS_TOKEN, _ADME_ACCESS_TOKEN_EXPIRES_ON
    if _ADME_ACCESS_TOKEN and _ADME_ACCESS_TOKEN_EXPIRES_ON > int(time()) + 300:
        return _ADME_ACCESS_TOKEN
    _ADME_ACCESS_TOKEN, _ADME_ACCESS_TOKEN_EXPIRES_ON = _acquire_adme_access_token()
    return _ADME_ACCESS_TOKEN


def _schema_doc_cache_key(kind: str) -> tuple[str, str, str]:
    endpoint, data_partition_id, _ = _adme_schema_config()
    return endpoint, data_partition_id, kind


def _adme_schema_url(kind: str) -> str:
    endpoint, _, _ = _adme_schema_config()
    path = globals().get("ADME_SCHEMA_SERVICE_PATH", "/api/schema-service/v1/schema")
    return f"{endpoint}{path}/{quote(kind, safe=':.-_')}"


def _adme_schema_source_prefix() -> str:
    endpoint, data_partition_id, _ = _adme_schema_config()
    return f"adme:endpoint={endpoint};partition={data_partition_id};"


def _adme_schema_source(schema_url: str) -> str:
    return f"{_adme_schema_source_prefix()}url={schema_url}"


_JSON_SCHEMA_MARKER_KEYS = {
    "$schema",
    "$id",
    "allOf",
    "anyOf",
    "definitions",
    "oneOf",
    "properties",
    "type",
    "x-osdu-schema-source",
}


def _looks_like_json_schema(value: Any) -> bool:
    return isinstance(value, dict) and any(key in value for key in _JSON_SCHEMA_MARKER_KEYS)


def _coerce_schema_body(value: Any, kind: str, context: str) -> dict[str, Any]:
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError as exc:
            raise ValueError(f"ADME schema service {context} for {kind} is not valid JSON.") from exc
    if not isinstance(value, dict):
        raise ValueError(f"ADME schema service {context} for {kind} is not a JSON object.")
    if not _looks_like_json_schema(value):
        raise ValueError(f"ADME schema service {context} for {kind} does not contain a JSON Schema body.")
    return value


def _extract_adme_schema_doc(response_doc: Any, kind: str) -> dict[str, Any]:
    if not isinstance(response_doc, dict):
        raise ValueError(f"ADME schema service response for {kind} is not a JSON object.")

    for field_name in ("schema", "schemaDocument", "schemaBody"):
        if field_name in response_doc:
            return _coerce_schema_body(response_doc[field_name], kind, f"response field {field_name!r}")

    if _looks_like_json_schema(response_doc):
        return response_doc

    raise ValueError(f"ADME schema service response for {kind} does not contain a JSON Schema body.")


def _adme_schema_headers() -> dict[str, str]:
    _, data_partition_id, token_scope = _adme_schema_config()
    try:
        token = get_adme_access_token()
    except Exception as exc:
        raise RuntimeError(f"Could not acquire ADME schema service token for scope {token_scope!r} using {_adme_auth_method()} authentication.") from exc
    return {
        "data-partition-id": data_partition_id,
        "Accept": "*/*",
        "Authorization": f"Bearer {token}",
    }


def _adme_schema_list_url(limit: int = 1) -> str:
    endpoint, _, _ = _adme_schema_config()
    path = globals().get("ADME_SCHEMA_SERVICE_PATH", "/api/schema-service/v1/schema")
    safe_limit = max(1, int(limit))
    return f"{endpoint}{path}?latestVersion=False&limit={safe_limit}"


def _adme_response_detail(resp) -> str:
    detail = (getattr(resp, "text", "") or "").strip().replace("\n", " ")
    return f"{detail[:1000]}..." if len(detail) > 1000 else detail


def _adme_schema_timeout(timeout: int | None = None) -> int:
    return int(timeout or globals().get("adme_schema_timeout_seconds", 30))


def _adme_schema_retry_status_codes() -> list[int]:
    return [int(code) for code in globals().get("adme_schema_retry_status_codes", [408, 429, 500, 502, 503, 504])]


def _adme_schema_retry_summary() -> str:
    return (
        f"total={int(globals().get('adme_schema_retry_total', 3))}, "
        f"backoff={float(globals().get('adme_schema_retry_backoff_seconds', 1.0))}, "
        f"statuses={_adme_schema_retry_status_codes()}"
    )


def _adme_schema_session() -> requests.Session:
    retry_total = max(0, int(globals().get("adme_schema_retry_total", 3)))
    backoff = max(0.0, float(globals().get("adme_schema_retry_backoff_seconds", 1.0)))
    retry = Retry(
        total=retry_total,
        connect=retry_total,
        read=retry_total,
        status=retry_total,
        backoff_factor=backoff,
        status_forcelist=_adme_schema_retry_status_codes(),
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def _adme_schema_get_json(url: str, context: str, timeout: int | None = None) -> Any:
    effective_timeout = _adme_schema_timeout(timeout)
    try:
        with _adme_schema_session() as session:
            resp = session.get(url, headers=_adme_schema_headers(), timeout=effective_timeout)
    except requests.RequestException as exc:
        raise requests.RequestException(
            f"ADME schema service request failed for {context} at {url} after retries ({_adme_schema_retry_summary()}): {exc}"
        ) from exc

    if not resp.ok:
        detail = _adme_response_detail(resp)
        message = f"ADME schema service returned HTTP {resp.status_code} for {context} at {url} after retries ({_adme_schema_retry_summary()})"
        if detail:
            message = f"{message}: {detail}"
        raise requests.HTTPError(message, response=resp)

    try:
        return resp.json()
    except ValueError as exc:
        raise ValueError(f"ADME schema service returned non-JSON content for {context} at {url}.") from exc


def validate_adme_schema_service_access(timeout: int | None = None) -> str:
    """Confirm the configured identity can read schemas from the ADME schema service."""
    list_url = _adme_schema_list_url(limit=1)
    payload = _adme_schema_get_json(list_url, "schema list probe", timeout=timeout)

    if isinstance(payload, list):
        return f"schema list probe succeeded ({len(payload)} item(s) returned; retries {_adme_schema_retry_summary()})"
    if isinstance(payload, dict):
        for field_name in ("schemas", "schemaInfos", "results", "data"):
            field_value = payload.get(field_name)
            if isinstance(field_value, list):
                return f"schema list probe succeeded ({len(field_value)} item(s) returned; retries {_adme_schema_retry_summary()})"
        return f"schema list probe succeeded (retries {_adme_schema_retry_summary()})"
    return f"schema list probe succeeded (retries {_adme_schema_retry_summary()})"


def _fetch_schema_doc_from_adme(kind: str, timeout: int | None = None) -> tuple[dict[str, Any], str]:
    schema_url = _adme_schema_url(kind)
    response_doc = _adme_schema_get_json(schema_url, kind, timeout=timeout)
    schema_doc = _extract_adme_schema_doc(response_doc, kind)
    return schema_doc, _adme_schema_source(schema_url)


def load_schema_doc(kind: str, timeout: int = 30) -> tuple[dict[str, Any], str]:
    cache_key = _schema_doc_cache_key(kind)
    if cache_key in _SCHEMA_DOC_CACHE:
        logger.info("Using cached schema for %s", kind)
        return _SCHEMA_DOC_CACHE[cache_key]

    cached = _load_schema_doc_from_persistent_cache(kind)
    if cached is not None:
        logger.info("Using persisted schema cache for %s", kind)
        _SCHEMA_DOC_CACHE[cache_key] = cached
        return cached

    schema_doc, source = _fetch_schema_doc_from_adme(kind, timeout=timeout)
    schema_result = (schema_doc, source)
    _SCHEMA_DOC_CACHE[cache_key] = schema_result
    _write_schema_doc_to_persistent_cache(kind, schema_doc, source)
    return schema_result


def load_schema_docs(kinds: list[str], timeout: int = 30) -> dict[str, tuple[dict[str, Any], str]]:
    unique_kinds = list(dict.fromkeys(kind for kind in kinds if kind))
    schema_results: dict[str, tuple[dict[str, Any], str]] = {}
    missing_kinds: list[str] = []

    for kind in unique_kinds:
        cache_key = _schema_doc_cache_key(kind)
        if cache_key in _SCHEMA_DOC_CACHE:
            logger.info("Using cached schema for %s", kind)
            schema_results[kind] = _SCHEMA_DOC_CACHE[cache_key]
        else:
            missing_kinds.append(kind)

    persisted = _load_schema_docs_from_persistent_cache(missing_kinds)
    for kind, schema_result in persisted.items():
        logger.info("Using persisted schema cache for %s", kind)
        _SCHEMA_DOC_CACHE[_schema_doc_cache_key(kind)] = schema_result
        schema_results[kind] = schema_result

    fetch_kinds = [kind for kind in missing_kinds if kind not in persisted]
    cache_rows: list[tuple[str, dict[str, Any], str]] = []
    for kind in fetch_kinds:
        try:
            schema_doc, source = _fetch_schema_doc_from_adme(kind, timeout=timeout)
            schema_result = (schema_doc, source)
            _SCHEMA_DOC_CACHE[_schema_doc_cache_key(kind)] = schema_result
            schema_results[kind] = schema_result
            cache_rows.append((kind, schema_doc, source))
            logger.info("Loaded schema for %s (%s)", kind, source)
        except Exception as exc:
            _SCHEMA_LOAD_ERRORS[kind] = str(exc)
            logger.warning("Failed to load schema for %s: %s", kind, exc)

    _write_schema_docs_to_persistent_cache(cache_rows)
    return schema_results


# ── naming.child_table_name ───────────────────────────────────────────


def child_table_name(parent: str, array_field: str) -> str:
    """Build child table name using triple-underscore separator."""
    return f"{parent}___{array_field}"


# ── types.DecomposedKind ──────────────────────────────────────────────


@dataclass
class DecomposedKind:
    """Result of decomposing one OSDU kind into parent + children."""

    kind_name: str
    col_classes: dict[str, list[str]]
    parent: Any  # DataFrame
    children: dict[str, tuple[str, Any]]  # table_name -> (source_col, df)


# ── schema_registry.py ────────────────────────────────────────────────
#    minus from_spark / from_delta / from_dataframe (Delta-table loaders unused here) ──

_JSON_TYPE_MAP: dict[str, T.DataType] = {
    "string": T.StringType(),
    "integer": T.LongType(),
    "number": T.DoubleType(),
    "boolean": T.BooleanType(),
}


def _resolve_node(
    node: dict[str, Any],
    definitions: dict[str, Any] | None,
) -> dict[str, Any]:
    if not isinstance(node, dict):
        return node

    if "$ref" in node and definitions:
        ref = node["$ref"]
        if ref.startswith("#/definitions/"):
            resolved = definitions.get(ref[len("#/definitions/") :])
            if resolved:
                return _resolve_node(resolved, definitions)

    if "allOf" in node:
        merged: dict[str, Any] = {}
        for sub in node["allOf"]:
            if not isinstance(sub, dict):
                continue
            resolved = _resolve_node(sub, definitions)
            if "properties" in resolved:
                merged.update(resolved["properties"])
        if merged:
            return {"type": "object", "properties": merged}

    return node


def _json_schema_to_spark(
    schema_node: dict[str, Any],
    definitions: dict[str, Any] | None = None,
) -> T.DataType:
    if not isinstance(schema_node, dict):
        return T.StringType()

    json_type = schema_node.get("type")

    if "$ref" in schema_node and not json_type:
        if definitions:
            ref = schema_node["$ref"]
            if ref.startswith("#/definitions/"):
                def_key = ref[len("#/definitions/") :]
                resolved = definitions.get(def_key)
                if resolved:
                    return _json_schema_to_spark(resolved, definitions)
        return T.StringType()

    if "allOf" in schema_node and not json_type:
        merged_props: dict[str, Any] = {}
        for sub in schema_node["allOf"]:
            if not isinstance(sub, dict):
                continue
            resolved = sub
            if "$ref" in sub and definitions:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    r = definitions.get(ref[len("#/definitions/") :])
                    if r:
                        resolved = r
            if "properties" in resolved:
                merged_props.update(resolved["properties"])
        if merged_props:
            fields = []
            for name, prop_schema in merged_props.items():
                spark_type = _json_schema_to_spark(prop_schema, definitions)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields)
        for sub in schema_node["allOf"]:
            if isinstance(sub, dict) and ("type" in sub or "properties" in sub):
                return _json_schema_to_spark(sub, definitions)
        return T.StringType()

    for composite_key in ("anyOf", "oneOf"):
        if composite_key in schema_node and not json_type:
            for sub in schema_node[composite_key]:
                if isinstance(sub, dict) and ("type" in sub or "properties" in sub):
                    return _json_schema_to_spark(sub, definitions)
            return T.StringType()

    if json_type == "object":
        props = schema_node.get("properties", {})
        if props:
            fields = []
            for name, prop_schema in props.items():
                spark_type = _json_schema_to_spark(prop_schema, definitions)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields)
        return T.MapType(T.StringType(), T.StringType())

    if json_type == "array":
        items = schema_node.get("items", {})
        element_type = _json_schema_to_spark(items, definitions)
        return T.ArrayType(element_type, containsNull=True)

    if json_type in _JSON_TYPE_MAP:
        return _JSON_TYPE_MAP[json_type]

    fmt = schema_node.get("format", "")
    if fmt in ("date-time", "date"):
        return T.StringType()

    return T.StringType()


def _classify_spark_type(dt: T.DataType) -> str:
    if isinstance(dt, T.ArrayType):
        return "json_array"
    if isinstance(dt, (T.StructType, T.MapType)):
        return "json_object"
    return "scalar"


def _parse_osdu_schema(schema_json: str | dict) -> dict[str, Any]:
    raw = json.loads(schema_json) if isinstance(schema_json, str) else schema_json

    definitions = raw.get("definitions", {})
    top_props = dict(raw.get("properties", {}))

    if "allOf" in raw:
        for sub in raw["allOf"]:
            if not isinstance(sub, dict):
                continue
            if "properties" in sub:
                top_props.update(sub["properties"])
            elif "$ref" in sub:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    def_key = ref[len("#/definitions/") :]
                    defn = definitions.get(def_key, {})
                    def_props = defn.get("properties", {})
                    top_props.update(def_props)

    envelope_fields = {k: v for k, v in top_props.items() if k != "data"}
    data_node = top_props.get("data", {})
    data_fields = dict(data_node.get("properties", {}))

    if "allOf" in data_node:
        for sub in data_node["allOf"]:
            if not isinstance(sub, dict):
                continue
            if "properties" in sub:
                data_fields.update(sub["properties"])
            elif "$ref" in sub:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    def_key = ref[len("#/definitions/") :]
                    defn = definitions.get(def_key, {})
                    def_props = defn.get("properties", {})
                    data_fields.update(def_props)

    return {
        "envelope_fields": envelope_fields,
        "data_fields": data_fields,
        "definitions": definitions,
        "raw": raw,
    }


class SchemaRegistry:
    """In-memory registry of OSDU schemas for schema-driven decomposition."""

    def __init__(self, schemas: dict[str, dict[str, Any]]) -> None:
        self._schemas = schemas
        logger.info("SchemaRegistry loaded: %d kinds", len(schemas))

    @classmethod
    def from_dict(cls, raw_schemas: dict[str, str | dict]) -> "SchemaRegistry":
        schemas = {}
        for kind, schema in raw_schemas.items():
            try:
                parsed = _parse_osdu_schema(schema)
                schemas[kind] = parsed
            except (json.JSONDecodeError, KeyError, TypeError) as e:
                logger.warning("Failed to parse schema for %s: %s", kind, e)
        return cls(schemas)

    @property
    def kinds(self) -> list[str]:
        return list(self._schemas.keys())

    def has_kind(self, kind: str) -> bool:
        return kind in self._schemas

    def has_field(self, kind: str, field_path: str) -> bool:
        info = self._schemas.get(kind)
        if info is None:
            return False
        if field_path in info["envelope_fields"]:
            return True
        clean = field_path[5:] if field_path.startswith("data.") else field_path
        return clean in info["data_fields"]

    def data_fields(self, kind: str) -> dict[str, Any]:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")
        return info["data_fields"]

    def classify_kind(
        self,
        kind: str,
        actual_columns: list[str] | None = None,
    ) -> dict[str, list[str]]:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")

        data_fields = info["data_fields"]
        envelope_fields = info["envelope_fields"]

        result: dict[str, list[str]] = {
            "scalar": [],
            "json_array": [],
            "json_object": [],
            "null": [],
        }

        field_lookup: dict[str, dict] = {}

        data_node = info["raw"].get("properties", {}).get("data", {})
        if not data_node and "allOf" in info["raw"]:
            for sub in info["raw"]["allOf"]:
                if isinstance(sub, dict) and "properties" in sub and "data" in sub["properties"]:
                    data_node = sub["properties"]["data"]
                    break
        if data_node:
            field_lookup["data"] = data_node

        field_lookup.update(envelope_fields)

        for name, schema_node in data_fields.items():
            field_lookup[f"data.{name}"] = schema_node
            field_lookup[name] = schema_node

        columns_to_classify = actual_columns if actual_columns is not None else list(field_lookup.keys())

        for col in columns_to_classify:
            schema_node = field_lookup.get(col)

            if schema_node is None and "." in col:
                path = col
                if path.startswith("data."):
                    path = path[5:]
                parts = path.split(".")
                current = data_fields
                resolved = None
                for i, part in enumerate(parts):
                    node = current.get(part)
                    if node is None:
                        break
                    node = _resolve_node(node, info.get("definitions"))
                    if i == len(parts) - 1:
                        resolved = node
                    else:
                        current = node.get("properties", {})
                if resolved is not None:
                    schema_node = resolved

            if schema_node is None:
                result["scalar"].append(col)
                continue

            spark_type = _json_schema_to_spark(schema_node, info.get("definitions"))
            category = _classify_spark_type(spark_type)
            result[category].append(col)

        return result

    def spark_schema_for_field(
        self,
        kind: str,
        field_path: str,
    ) -> T.DataType | None:
        info = self._schemas.get(kind)
        if info is None:
            return None

        if field_path == "data":
            defs = info.get("definitions")
            fields = []
            for name, schema_node in info["data_fields"].items():
                spark_type = _json_schema_to_spark(schema_node, defs)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields) if fields else None

        clean_path = field_path
        if clean_path.startswith("data."):
            clean_path = clean_path[5:]

        parts = clean_path.split(".")
        current = info["data_fields"]
        defs = info.get("definitions")
        for i, part in enumerate(parts):
            if part not in current:
                if i == 0 and part in info["envelope_fields"]:
                    current = info["envelope_fields"]
                else:
                    return None

            node = current[part]
            node = _resolve_node(node, defs)
            if i < len(parts) - 1:
                if node.get("type") == "object":
                    current = node.get("properties", {})
                elif node.get("type") == "array":
                    items = node.get("items", {})
                    items = _resolve_node(items, defs)
                    current = items.get("properties", {})
                else:
                    return None
            else:
                return _json_schema_to_spark(node, defs)

        return None

    def spark_struct_for_kind(self, kind: str) -> T.StructType:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")

        defs = info.get("definitions")
        fields = []
        for name, schema_node in info["data_fields"].items():
            spark_type = _json_schema_to_spark(schema_node, defs)
            fields.append(T.StructField(name, spark_type, nullable=True))

        return T.StructType(fields)


def _schema_load_error(kind: str) -> str | None:
    return _SCHEMA_LOAD_ERRORS.get(kind)


def build_registry_from_adme(kinds: list[str]) -> SchemaRegistry:
    """Fetch OSDU schemas via the configured ADME schema service and build a SchemaRegistry.

    This is the standalone substitute for SchemaRegistry.from_spark(spark, "osdu_schemas").
    """
    raw: dict[str, dict] = {}
    unique_kinds = list(dict.fromkeys(kinds))
    for kind in unique_kinds:
        _SCHEMA_LOAD_ERRORS.pop(kind, None)

    schema_results = load_schema_docs(unique_kinds)
    for kind in unique_kinds:
        schema_result = schema_results.get(kind)
        if schema_result is None:
            continue
        doc, src = schema_result
        raw[kind] = doc
        logger.info("Loaded schema for %s (%s)", kind, src)
    return SchemaRegistry.from_dict(raw)


# ── decompose.py ──────────────────────────────────────────────────────

_FLATTEN_SAMPLE_SIZE = 10
_FLATTEN_PASS_LIMIT = 10
_DATA_PREFIX = "data__"
_ENVELOPE_FIELD_NAMES = {"data", "meta", "id", "version", "kind", "acl", "legal", "tags", "createUser", "createTime", "ingestTime", "fileDownloadTime", "fileDownloadState", "fileDownloadFolder", "isActive"}


def _merge_struct_types(a: T.StructType, b: T.StructType) -> T.StructType:
    fields_by_name: dict[str, T.StructField] = {f.name: f for f in a.fields}
    for field in b.fields:
        if field.name not in fields_by_name:
            fields_by_name[field.name] = field
        else:
            existing = fields_by_name[field.name]
            if isinstance(existing.dataType, T.NullType):
                fields_by_name[field.name] = field
            elif isinstance(field.dataType, T.NullType):
                pass
            elif isinstance(existing.dataType, T.StructType) and isinstance(field.dataType, T.StructType):
                merged = _merge_struct_types(existing.dataType, field.dataType)
                fields_by_name[field.name] = T.StructField(field.name, merged, nullable=True)
    return T.StructType(list(fields_by_name.values()))


def _merge_schemas(a: T.DataType, b: T.DataType) -> T.DataType:
    if isinstance(a, T.StructType) and isinstance(b, T.StructType):
        return _merge_struct_types(a, b)
    if isinstance(a, T.ArrayType) and isinstance(b, T.ArrayType):
        merged_elem = _merge_schemas(a.elementType, b.elementType)
        return T.ArrayType(merged_elem, containsNull=True)
    if isinstance(a, T.NullType):
        return b
    return a


def _normalize_json_sample_value(val: str, array_elements: bool = False) -> str | None:
    if not (val and isinstance(val, str) and val.strip() and val.strip()[0] in "{["):
        return None
    try:
        if array_elements:
            parsed = json.loads(val)
            if not (isinstance(parsed, list) and parsed and isinstance(parsed[0], dict)):
                return None
            val = json.dumps(parsed[0])
        json.loads(val)
        return val
    except Exception:
        return None


def _infer_json_schemas_from_values(spark, values: list[str]) -> list[T.DataType | None]:
    if not values:
        return []
    try:
        schema_exprs = [F.schema_of_json(F.lit(v)).alias(f"_schema_{i}") for i, v in enumerate(values)]
        row = spark.range(1).select(*schema_exprs).collect()[0]
        inferred: list[T.DataType | None] = []
        for i in range(len(values)):
            schema_str = row[f"_schema_{i}"]
            inferred.append(T._parse_datatype_string(schema_str) if schema_str else None)
        return inferred
    except Exception:
        return [None] * len(values)


def _infer_json_schema_from_value(spark, val: str, array_elements: bool = False) -> T.DataType | None:
    normalized = _normalize_json_sample_value(val, array_elements=array_elements)
    if normalized is None:
        return None
    return _infer_json_schemas_from_values(spark, [normalized])[0]


def _infer_json_schema(
    df: DataFrame,
    col_name: str,
    sample_size: int = _FLATTEN_SAMPLE_SIZE,
    array_elements: bool = False,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
    cache_scope: str = "",
) -> T.DataType | None:
    cache_key = (cache_scope, col_name, array_elements)
    if schema_cache is not None and cache_key in schema_cache:
        return schema_cache[cache_key]

    non_null = df.filter(F.col(col_name).isNotNull()).select(col_name)
    samples = non_null.limit(sample_size).collect()
    if not samples:
        if schema_cache is not None:
            schema_cache[cache_key] = None
        return None

    spark = df.sparkSession
    merged: T.DataType | None = None

    normalized_values: list[str] = []
    for row in samples:
        normalized = _normalize_json_sample_value(row[0], array_elements=array_elements)
        if normalized is not None:
            normalized_values.append(normalized)

    if normalized_values:
        inferred_schemas = _infer_json_schemas_from_values(spark, normalized_values)
        for sample_schema in inferred_schemas:
            if sample_schema is None:
                continue
            merged = sample_schema if merged is None else _merge_schemas(merged, sample_schema)

    if schema_cache is not None:
        schema_cache[cache_key] = merged
    return merged


_ENVELOPE_ALIASES: list[tuple[str, str, bool]] = [
    ("createTime", "create_time", True),
    ("modifyTime", "modify_time", True),
    ("createUser", "create_user", False),
    ("modifyUser", "modify_user", False),
]

_ENVELOPE_RENAMES: list[tuple[str, str]] = []  # OSDU-native id/kind/version preserved


def _extract_envelope(df: DataFrame) -> DataFrame:
    cols = set(df.columns)

    for src, tgt in _ENVELOPE_RENAMES:
        if src in cols and tgt not in cols:
            df = df.withColumnRenamed(src, tgt)
            cols.discard(src)
            cols.add(tgt)

    for camel, snake, cast_ts in _ENVELOPE_ALIASES:
        if camel in cols:
            expr = F.col(camel).cast(T.TimestampType()) if cast_ts else F.col(camel)
            df = df.withColumn(snake, expr)
            if camel != snake:
                df = df.drop(camel)
                cols.discard(camel)
                cols.add(snake)
        elif snake in cols and cast_ts:
            df = df.withColumn(snake, F.col(snake).cast(T.TimestampType()))

    df = df.withColumn("ingested_at", F.current_timestamp())
    return df


def classify_columns(
    df: DataFrame,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> dict[str, list[str]]:
    if registry is None or kind is None:
        raise ValueError(
            "SchemaRegistry and kind are required for classify_columns. Sampling-based classification has been removed."
        )

    dot_columns = [c.replace("__", ".") for c in df.columns]
    classified = registry.classify_kind(kind, actual_columns=dot_columns)

    unresolved = [
        col for col in dot_columns
        if col not in _ENVELOPE_FIELD_NAMES and col in classified.get("scalar", []) and not registry.has_field(kind, col)
    ]
    if unresolved:
        logger.warning(
            "Columns not found in registry for kind %r (classified as scalar): %s",
            kind,
            unresolved,
        )

    return {cat: [c.replace(".", "__") for c in cols] for cat, cols in classified.items()}


def flatten_json_object_col(
    df: DataFrame,
    col_name: str,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> DataFrame:
    col_type = df.schema[col_name].dataType
    if not isinstance(col_type, T.StringType):
        logger.warning(
            "Column %r classified as json_object but has type %s — skipping flatten",
            col_name,
            col_type.simpleString(),
        )
        return df

    schema = None
    if registry is not None and kind is not None:
        field_path = col_name.replace("__", ".")
        schema = registry.spark_schema_for_field(kind, field_path)
        if schema is not None and not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r registry-typed %s — will try sampling for struct schema",
                col_name,
                type(schema).__name__,
            )
            schema = None

    if schema is None:
        schema = _infer_json_schema(df, col_name)
        if schema is None or not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r — no struct schema from registry or sampling, deferring to loop",
                col_name,
            )
            return df

    parsed_alias = f"_parsed_{col_name.replace('__', '_')}"
    df = df.withColumn(parsed_alias, F.from_json(F.col(col_name), schema))
    for field in schema.fields:
        new_col = f"{col_name}__{field.name}"
        df = df.withColumn(new_col, F.col(f"{parsed_alias}.{field.name}"))
    df = df.drop(parsed_alias).drop(col_name)
    return df


def build_parent(
    df: DataFrame,
    col_classes: dict[str, list[str]],
    drop_wkt: bool = True,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> DataFrame:
    keep_cols = col_classes["scalar"] + col_classes["null"]
    parent_cols = list(dict.fromkeys(keep_cols + col_classes["json_object"]))
    parent = df.select([F.col(c) for c in parent_cols])

    for obj_col in col_classes["json_object"]:
        if obj_col in parent.columns:
            parent = flatten_json_object_col(parent, obj_col, registry=registry, kind=kind)

    if drop_wkt:
        drop_cols = [c for c in parent.columns if c.endswith("__wkt")]
        if drop_cols:
            parent = parent.drop(*drop_cols)

    parent = _extract_envelope(parent)
    return parent



def _merge_key_column_names(df: DataFrame) -> list[str]:
    configured = globals().get("merge_key_columns", ["id", "version"])
    return [col for col in configured if col in df.columns]


def _merge_key_fields(df: DataFrame) -> list[T.StructField]:
    fields = []
    for col in _merge_key_column_names(df):
        field = next((f for f in df.schema.fields if f.name == col), None)
        fields.append(T.StructField(col, field.dataType if field else T.StringType(), True))
    return fields


def _build_child_primitive(df: DataFrame, col_name: str) -> DataFrame:
    schema = T.ArrayType(T.StringType())
    key_cols = _merge_key_column_names(df)
    parsed = df.select(
        *key_cols,
        F.from_json(F.col(col_name), schema).alias("_items"),
    ).filter(F.col("_items").isNotNull())
    return parsed.select(
        *key_cols,
        F.posexplode("_items").alias("ordinal", "value"),
    )


def _build_child_struct(df: DataFrame, col_name: str, array_schema: T.ArrayType) -> DataFrame:
    key_cols = _merge_key_column_names(df)
    parsed = df.select(
        *key_cols,
        F.from_json(F.col(col_name), array_schema).alias("_items"),
    ).filter(F.col("_items").isNotNull())

    exploded = parsed.select(
        *key_cols,
        F.posexplode("_items").alias("ordinal", "_item"),
    )

    if isinstance(array_schema.elementType, T.StructType):
        select_cols = [*key_cols, "ordinal"]
        select_cols.extend(F.col(f"_item.{fld.name}").alias(fld.name) for fld in array_schema.elementType.fields)
        return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(exploded.select(select_cols)))
    else:
        return exploded.select(*key_cols, "ordinal", F.col("_item").alias("value"))


def _is_primitive_array(sample_json: str) -> bool:
    try:
        items = json.loads(sample_json)
        if not isinstance(items, list) or len(items) == 0:
            return False
        return not isinstance(items[0], dict)
    except (json.JSONDecodeError, TypeError):
        return False


def build_child_table(
    df: DataFrame,
    col_name: str,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
    sample_val: str | None = None,
) -> DataFrame:
    spark = df.sparkSession

    if sample_val is None:
        sample_row = df.filter(F.col(col_name).isNotNull()).select(col_name).limit(1).collect()
        if sample_row:
            sample_val = sample_row[0][0]

    if sample_val is None:
        if registry is not None and kind is not None:
            field_path = col_name.replace("__", ".")
            reg_type = registry.spark_schema_for_field(kind, field_path)
            if reg_type is not None:
                schema = reg_type if isinstance(reg_type, T.ArrayType) else T.ArrayType(reg_type)
                return _build_child_struct(df, col_name, schema)
        schema = T.StructType(
            [
                *_merge_key_fields(df),
                T.StructField("ordinal", T.IntegerType()),
            ]
        )
        return spark.createDataFrame([], schema)

    if _is_primitive_array(sample_val):
        return _build_child_primitive(df, col_name)

    if registry is None or kind is None:
        logger.info("No registry for %r — inferring schema from sample data", col_name)
        inferred = _infer_json_schema(
            df, col_name, schema_cache=schema_cache, cache_scope=f"child_array:{col_name}",
        )
        if inferred is not None:
            schema = inferred if isinstance(inferred, T.ArrayType) else T.ArrayType(inferred)
        else:
            schema = T.ArrayType(T.StringType())
    else:
        field_path = col_name.replace("__", ".")
        reg_type = registry.spark_schema_for_field(kind, field_path)
        if reg_type is not None and isinstance(reg_type, T.ArrayType):
            schema = reg_type
        elif reg_type is not None:
            schema = T.ArrayType(reg_type)
        else:
            inferred = _infer_json_schema(
                df, col_name, schema_cache=schema_cache, cache_scope=f"child_array:{col_name}",
            )
            if inferred is not None:
                logger.info("Column %r not in registry for %r — inferred schema from data", col_name, kind)
                schema = inferred if isinstance(inferred, T.ArrayType) else T.ArrayType(inferred)
            else:
                logger.warning("Column %r not in registry for %r and inference failed — using StringType", col_name, kind)
                schema = T.ArrayType(T.StringType())

    if not isinstance(schema, T.ArrayType):
        schema = T.ArrayType(schema)

    return _build_child_struct(df, col_name, schema)


def build_all_children(
    df: DataFrame,
    col_classes: dict[str, list[str]],
    kind_prefix: str,
    registry: SchemaRegistry | None = None,
    osdu_kind: str | None = None,
) -> dict[str, tuple[str, DataFrame]]:
    children: dict[str, tuple[str, DataFrame]] = {}
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] = {}

    json_array_cols_present = [c for c in col_classes["json_array"] if c in df.columns]
    sampled_values: dict[str, str | None] = {}
    has_items_by_col: dict[str, bool] = {}
    if json_array_cols_present:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in json_array_cols_present]
        sample_row = df.select(sample_exprs).collect()
        if sample_row:
            row = sample_row[0]
            sampled_values = {c: row[c] for c in json_array_cols_present}
        else:
            sampled_values = {c: None for c in json_array_cols_present}

        has_items_by_col = _batch_sample_json_array_presence(df, json_array_cols_present)

    for col_name in col_classes["json_array"]:
        if col_name in has_items_by_col and not has_items_by_col[col_name] and not globals().get("create_empty_child_tables", True):
            logger.info("Skipping empty json_array child %r (no non-empty arrays in batch)", col_name)
            continue

        normalized = col_name.replace(".", "__")
        if normalized.startswith(_DATA_PREFIX):
            normalized = normalized[len(_DATA_PREFIX) :]
        parts = [p for p in normalized.split("__") if p]
        table_suffix = "__".join(parts[-2:]) if len(parts) > 2 else "__".join(parts)

        table_name = child_table_name(kind_prefix, table_suffix)
        child_df = build_child_table(
            df, col_name, registry=registry, kind=osdu_kind,
            schema_cache=schema_cache, sample_val=sampled_values.get(col_name),
        )
        children[table_name] = (col_name, child_df)

    return children


_TAGS_COLUMN = "tags"


def extract_tags(df: DataFrame, kind_prefix: str) -> tuple[str, DataFrame] | None:
    if _TAGS_COLUMN not in df.columns:
        return None

    schema = T.MapType(T.StringType(), T.StringType())
    key_cols = _merge_key_column_names(df)
    parsed = df.select(
        *key_cols,
        F.from_json(F.col(_TAGS_COLUMN), schema).alias("_tags_map"),
    ).filter(F.col("_tags_map").isNotNull())

    if not parsed.take(1):
        return None

    child = parsed.select(
        *key_cols,
        F.explode("_tags_map").alias("tag_key", "tag_value"),
    )

    table_name = child_table_name(kind_prefix, "tags")
    return table_name, child


_NUMERIC_TYPES = (T.DoubleType, T.FloatType, T.LongType, T.IntegerType, T.ShortType)


def _coerce_id_columns(df: DataFrame) -> DataFrame:
    for field in df.schema.fields:
        if field.name.endswith("_id") and isinstance(field.dataType, _NUMERIC_TYPES):
            df = df.withColumn(field.name, F.col(field.name).cast(T.StringType()))
    return df


def _serialize_complex_columns(df: DataFrame) -> DataFrame:
    for field in df.schema.fields:
        if isinstance(field.dataType, T.MapType) or (
            isinstance(field.dataType, T.ArrayType) and not isinstance(field.dataType.elementType, T.StructType)
        ):
            df = df.withColumn(field.name, F.to_json(F.col(field.name)))
    return df


def _explode_typed_array(
    df: DataFrame,
    col_name: str,
    elem_type: T.DataType,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> DataFrame:
    key_cols = _merge_key_column_names(df)
    subset = df.select(*key_cols, F.col(col_name)).filter(F.col(col_name).isNotNull())
    exploded = subset.select(
        *key_cols,
        F.posexplode(F.col(col_name)).alias("ordinal", "_item"),
    )

    if isinstance(elem_type, T.StructType):
        select_cols = [*key_cols, "ordinal"]
        select_cols.extend(F.col(f"_item.{fld.name}").alias(fld.name) for fld in elem_type.fields)
        return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(exploded.select(select_cols)))

    if isinstance(elem_type, T.StringType):
        child = exploded.select(*key_cols, "ordinal", F.col("_item").alias("value"))
        inferred = _infer_json_schema(
            child, "value", schema_cache=schema_cache, cache_scope=f"typed_array:{col_name}",
        )
        if inferred is not None and isinstance(inferred, T.StructType):
            logger.info(
                "Column %r has StringType elements containing JSON objects — re-parsing with inferred schema (%d fields: %s)",
                col_name, len(inferred.fields), ", ".join(f.name for f in inferred.fields),
            )
            child = child.withColumn("_parsed", F.from_json(F.col("value"), inferred))
            select_cols: list = [*key_cols, "ordinal"]
            for fld in inferred.fields:
                select_cols.append(F.col(f"_parsed.{fld.name}").alias(fld.name))
            return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(child.select(select_cols)))
        return child

    return exploded.select(*key_cols, "ordinal", F.col("_item").alias("value"))


def _batch_sample_string_arrays(
    parent: DataFrame,
    fields: list[T.StructField],
) -> dict[str, bool]:
    if not fields:
        return {}

    sample_exprs = [
        F.first(
            F.when(F.size(F.col(f.name)) > 0, F.col(f.name).getItem(0)),
            ignorenulls=True,
        ).alias(f.name)
        for f in fields
    ]

    samples = parent.select(sample_exprs).collect()
    if not samples:
        return {f.name: False for f in fields}

    row = samples[0]
    results: dict[str, bool] = {}
    for field in fields:
        val = row[field.name]
        if not val or not isinstance(val, str):
            results[field.name] = False
            continue
        try:
            parsed = json.loads(val)
            results[field.name] = isinstance(parsed, dict)
        except (json.JSONDecodeError, ValueError):
            results[field.name] = False

    return results


def _batch_sample_array_presence(
    parent: DataFrame,
    fields: list[T.StructField],
) -> dict[str, bool]:
    if not fields:
        return {}

    sample_exprs = [
        F.first(
            F.when(F.size(F.col(f.name)) > 0, F.lit(1)),
            ignorenulls=True,
        ).alias(f.name)
        for f in fields
    ]

    samples = parent.select(sample_exprs).collect()
    if not samples:
        return {f.name: False for f in fields}

    row = samples[0]
    return {field.name: row[field.name] is not None for field in fields}


def _batch_sample_json_array_presence(
    df: DataFrame,
    columns: list[str],
) -> dict[str, bool]:
    if not columns:
        return {}

    sample_exprs = [
        F.first(
            F.when(
                F.col(c).isNotNull() & (F.length(F.regexp_replace(F.col(c), r"\s+", "")) > 2),
                F.lit(1),
            ),
            ignorenulls=True,
        ).alias(c)
        for c in columns
    ]

    samples = df.select(sample_exprs).collect()
    if not samples:
        return {c: False for c in columns}

    row = samples[0]
    return {c: row[c] is not None for c in columns}


def _extract_typed_arrays(
    parent: DataFrame,
    kind_prefix: str,
    retained_parent_arrays: set[str] | None = None,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> tuple[dict[str, tuple[str, DataFrame]], DataFrame, set[str]]:
    children: dict[str, tuple[str, DataFrame]] = {}
    retained_parent_arrays = set() if retained_parent_arrays is None else retained_parent_arrays

    array_cols = [f for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType)]

    string_array_fields = [
        f
        for f in array_cols
        if f.name not in retained_parent_arrays and isinstance(f.dataType.elementType, T.StringType)
    ]
    string_array_results = _batch_sample_string_arrays(parent, string_array_fields)

    extract_fields: list[T.StructField] = []
    candidate_extract_fields: list[T.StructField] = []
    for field in array_cols:
        if field.name in retained_parent_arrays:
            continue

        elem_type = field.dataType.elementType
        if isinstance(elem_type, T.StructType):
            is_child = True
        elif isinstance(elem_type, T.StringType):
            is_child = string_array_results.get(field.name, False)
        else:
            is_child = False

        if not is_child:
            retained_parent_arrays.add(field.name)
            logger.info(
                "Keeping primitive array %r on parent (element type: %s)",
                field.name, field.dataType.elementType.simpleString(),
            )
            continue

        candidate_extract_fields.append(field)

    has_items_by_col = _batch_sample_array_presence(parent, candidate_extract_fields)

    for field in candidate_extract_fields:
        if not has_items_by_col.get(field.name, False):
            logger.info("Skipping empty typed array %r (no non-empty values in batch)", field.name)
            continue

        extract_fields.append(field)
        col_name = field.name
        normalized = col_name
        if normalized.startswith(_DATA_PREFIX):
            normalized = normalized[len(_DATA_PREFIX) :]
        parts = [p for p in normalized.split("__") if p]
        table_suffix = "__".join(parts[-2:]) if len(parts) > 2 else "__".join(parts)

        table_name = child_table_name(kind_prefix, table_suffix)
        child_df = _explode_typed_array(parent, col_name, field.dataType.elementType, schema_cache=schema_cache)
        children[table_name] = (col_name, child_df)

    if extract_fields:
        parent = parent.drop(*[f.name for f in extract_fields])

    return children, parent, retained_parent_arrays


def _flatten_typed_structs(parent: DataFrame) -> DataFrame:
    struct_fields = [f for f in parent.schema.fields if isinstance(f.dataType, T.StructType)]
    if not struct_fields:
        return parent

    select_exprs = []
    struct_names = {f.name for f in struct_fields}
    for field in parent.schema.fields:
        if field.name in struct_names:
            prefix = field.name
            select_exprs.extend(
                F.col(f"{prefix}.{sub.name}").alias(f"{prefix}__{sub.name}") for sub in field.dataType.fields
            )
        else:
            select_exprs.append(F.col(field.name))

    return parent.select(select_exprs)


def _flatten_all_struct_columns(df: DataFrame) -> DataFrame:
    while any(isinstance(field.dataType, T.StructType) for field in df.schema.fields):
        df = _flatten_typed_structs(df)
    return df


def _flatten_inferred_json_string_columns(df: DataFrame) -> DataFrame:
    while True:
        string_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, T.StringType)]
        if not string_cols:
            return df

        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in string_cols]
        samples = df.select(sample_exprs).collect()
        row = samples[0] if samples else None

        flattened_any = False

        normalized_by_col: dict[str, str] = {}
        for col_name in string_cols:
            val = row[col_name] if row is not None else None
            normalized = _normalize_json_sample_value(val)
            if normalized is not None:
                normalized_by_col[col_name] = normalized

        inferred_by_col: dict[str, T.DataType | None] = {}
        if normalized_by_col:
            cols = list(normalized_by_col.keys())
            vals = [normalized_by_col[c] for c in cols]
            inferred = _infer_json_schemas_from_values(df.sparkSession, vals)
            inferred_by_col = {c: s for c, s in zip(cols, inferred, strict=False)}

        for col_name in string_cols:
            inferred = inferred_by_col.get(col_name)
            if inferred is None or not isinstance(inferred, T.StructType):
                continue

            logger.info(
                "Column %r contains embedded JSON objects — flattening inferred schema (%d fields: %s)",
                col_name, len(inferred.fields), ", ".join(field.name for field in inferred.fields),
            )
            df = df.withColumn("_parsed", F.from_json(F.col(col_name), inferred))
            for field in inferred.fields:
                df = df.withColumn(f"{col_name}__{field.name}", F.col(f"_parsed.{field.name}"))
            df = df.drop(col_name, "_parsed")
            flattened_any = True
            break

        if not flattened_any:
            return df


def _flatten_remaining_json_objects(
    parent: DataFrame,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
    _sampled_cache: dict[str, str | None] | None = None,
    _schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> tuple[DataFrame, bool]:
    string_cols = [f.name for f in parent.schema.fields if isinstance(f.dataType, T.StringType)]
    if not string_cols:
        return parent, False

    if _sampled_cache is None:
        _sampled_cache = {}

    new_cols = [c for c in string_cols if c not in _sampled_cache]
    if new_cols:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in new_cols]
        samples = parent.select(sample_exprs).collect()
        if samples:
            row = samples[0]
            for col_name in new_cols:
                _sampled_cache[col_name] = row[col_name]
        else:
            for col_name in new_cols:
                _sampled_cache[col_name] = None

    json_obj_cols = []
    json_array_cols = []
    for col_name in string_cols:
        val = _sampled_cache.get(col_name)
        if not val or not isinstance(val, str):
            continue
        stripped = val.strip()
        if stripped.startswith("{"):
            try:
                json.loads(val)
                json_obj_cols.append(col_name)
            except (json.JSONDecodeError, ValueError):
                pass
        elif stripped.startswith("[{"):
            try:
                parsed = json.loads(val)
                if parsed and isinstance(parsed[0], dict):
                    json_array_cols.append(col_name)
            except (json.JSONDecodeError, ValueError):
                pass

    if not json_obj_cols and not json_array_cols:
        return parent, False

    flattened_json_objects = False

    cols_to_drop = []
    parsed_col_defs: list[tuple[str, str, T.StructType]] = []

    for col_name in json_obj_cols:
        schema = None
        sampled_val = _sampled_cache.get(col_name)
        if registry is not None and kind is not None:
            field_path = col_name.replace("__", ".")
            schema = registry.spark_schema_for_field(kind, field_path)

        if schema is not None and not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r is registry-typed %s but contains JSON objects — inferring schema from data",
                col_name, type(schema).__name__,
            )
            schema = _infer_json_schema_from_value(parent.sparkSession, sampled_val)

        if schema is None:
            schema = _infer_json_schema_from_value(parent.sparkSession, sampled_val)

        if schema is None or not isinstance(schema, T.StructType):
            logger.warning(
                "Nested JSON in %r not resolvable from registry or sampling for kind %r — skipping",
                col_name, kind,
            )
            continue

        safe_suffix = col_name.split("__")[-1]
        parsed_alias = f"_parsed_{safe_suffix}_{len(parsed_col_defs)}"
        parsed_col_defs.append((col_name, parsed_alias, schema))
        cols_to_drop.append(col_name)

    if parsed_col_defs:
        for col_name, parsed_alias, schema in parsed_col_defs:
            parent = parent.withColumn(parsed_alias, F.from_json(F.col(col_name), schema))

        select_exprs = []
        drop_set = set(cols_to_drop) | {alias for _, alias, _ in parsed_col_defs}
        select_exprs.extend(F.col(field.name) for field in parent.schema.fields if field.name not in drop_set)

        for col_name, parsed_alias, schema in parsed_col_defs:
            for field in schema.fields:
                new_col = f"{col_name}__{field.name}"
                select_exprs.append(F.col(f"{parsed_alias}.{field.name}").alias(new_col))

        parent = parent.select(select_exprs)
        flattened_json_objects = True

    for col_name in json_array_cols:
        schema = None
        sampled_val = _sampled_cache.get(col_name)
        if registry is not None and kind is not None:
            field_path = col_name.replace("__", ".")
            schema = registry.spark_schema_for_field(kind, field_path)

        elem_schema = None
        if schema is not None and isinstance(schema, T.ArrayType):
            if isinstance(schema.elementType, T.StructType):
                elem_schema = schema.elementType
        elif schema is not None and isinstance(schema, T.StructType):
            elem_schema = schema

        if elem_schema is None:
            inferred = _infer_json_schema_from_value(parent.sparkSession, sampled_val, array_elements=True)
            if inferred is not None and isinstance(inferred, T.StructType):
                elem_schema = inferred

        if elem_schema is None:
            logger.warning(
                "JSON array in %r not resolvable from registry or sampling for kind %r — skipping",
                col_name, kind,
            )
            continue

        array_schema = T.ArrayType(elem_schema)
        parent = parent.withColumn(col_name, F.from_json(F.col(col_name), array_schema))
        logger.info(
            "Parsed JSON array string %r → ArrayType(StructType(%d fields))",
            col_name, len(elem_schema.fields),
        )
        flattened_json_objects = True

    return parent, flattened_json_objects


def decompose_kind(
    df: DataFrame,
    kind_name: str,
    drop_wkt: bool = True,
    registry: SchemaRegistry | None = None,
    osdu_kind: str | None = None,
    input_rows: int | None = None,
    infer_nested_json: bool = True,
) -> DecomposedKind:
    t_start = perf_counter()
    stage_times: dict[str, float] = {}

    def _record_stage(name: str, started_at: float) -> None:
        stage_times[name] = stage_times.get(name, 0.0) + (perf_counter() - started_at)

    t_stage = perf_counter()
    col_classes = classify_columns(df, registry=registry, kind=osdu_kind)
    _record_stage("classify_columns", t_stage)
    row_info = str(input_rows) if input_rows is not None else "unknown"
    logger.info(
        "Decomposing kind=%s  rows=%s  cols=%d  (scalar=%d, json_array=%d, json_object=%d, null=%d)",
        kind_name, row_info, len(df.columns),
        len(col_classes["scalar"]), len(col_classes["json_array"]),
        len(col_classes["json_object"]), len(col_classes["null"]),
    )

    t_stage = perf_counter()
    if registry is not None:
        json_cols = col_classes["json_object"] + col_classes["json_array"]
        for col_name in json_cols:
            if col_name in df.columns:
                col_type = df.schema[col_name].dataType
                if not isinstance(col_type, T.StringType):
                    logger.info(
                        "Casting %r from %s → StringType (registry says json)",
                        col_name, col_type.simpleString(),
                    )
                    df = df.withColumn(col_name, F.col(col_name).cast(T.StringType()))
    _record_stage("coerce_json_columns", t_stage)

    if _TAGS_COLUMN in col_classes["json_object"]:
        col_classes["json_object"].remove(_TAGS_COLUMN)
    if _TAGS_COLUMN in col_classes["scalar"]:
        col_classes["scalar"].remove(_TAGS_COLUMN)

    t_stage = perf_counter()
    json_array_cols_present = [c for c in col_classes["json_array"] if c in df.columns]
    if json_array_cols_present:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in json_array_cols_present]
        sample_row = df.select(sample_exprs).collect()
        primitive_arrays: list[str] = []
        if sample_row:
            row = sample_row[0]
            for col_name in json_array_cols_present:
                val = row[col_name]
                if val and _is_primitive_array(val):
                    primitive_arrays.append(col_name)
        if primitive_arrays:
            col_classes["json_array"] = [c for c in col_classes["json_array"] if c not in primitive_arrays]
            col_classes["scalar"].extend(primitive_arrays)
            logger.info(
                "Reclassified %d primitive array column(s) as scalar: %s",
                len(primitive_arrays), primitive_arrays,
            )
    _record_stage("reclassify_primitive_arrays", t_stage)

    t_stage = perf_counter()
    parent = build_parent(df, col_classes, drop_wkt=drop_wkt, registry=registry, kind=osdu_kind)
    _record_stage("build_parent", t_stage)

    t_stage = perf_counter()
    children = build_all_children(df, col_classes, kind_prefix=kind_name, registry=registry, osdu_kind=osdu_kind)
    _record_stage("build_all_children", t_stage)
    retained_parent_arrays: set[str] = set()

    _sampled_cache: dict[str, str | None] = {}
    _schema_cache: dict[tuple[str, str, bool], T.DataType | None] = {}

    t_stage = perf_counter()
    typed_children, parent, retained_parent_arrays = _extract_typed_arrays(
        parent, kind_name,
        retained_parent_arrays=retained_parent_arrays,
        schema_cache=_schema_cache,
    )
    _record_stage("extract_typed_arrays_initial", t_stage)
    children.update(typed_children)

    _seen_array_cols: set[str] = set(f.name for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType))

    reached_flatten_pass_limit = False
    pass_index = 0
    while True:
        t_iter = perf_counter()
        made_progress = False

        t_struct = perf_counter()
        had_structs = any(isinstance(f.dataType, T.StructType) for f in parent.schema.fields)
        parent = _flatten_typed_structs(parent)
        _record_stage("flatten_typed_structs", t_struct)
        if had_structs:
            made_progress = True

        if infer_nested_json:
            t_json = perf_counter()
            parent, flattened_json = _flatten_remaining_json_objects(
                parent, registry=registry, kind=osdu_kind,
                _sampled_cache=_sampled_cache, _schema_cache=_schema_cache,
            )
            _record_stage("flatten_remaining_json_objects", t_json)
            if flattened_json:
                made_progress = True

        current_array_cols = set(f.name for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType))
        new_array_cols = current_array_cols - _seen_array_cols
        _seen_array_cols = current_array_cols

        if new_array_cols:
            t_new_arrays = perf_counter()
            new_children, parent, retained_parent_arrays = _extract_typed_arrays(
                parent, kind_name,
                retained_parent_arrays=retained_parent_arrays,
                schema_cache=_schema_cache,
            )
            _record_stage("extract_typed_arrays_loop", t_new_arrays)
            if new_children:
                children.update(new_children)
                made_progress = True

        iter_elapsed = perf_counter() - t_iter
        pass_index += 1
        logger.info(
            "Flatten iteration %d/%d for %s took %.3fs (progress=%s, new_arrays=%d)",
            pass_index, _FLATTEN_PASS_LIMIT, kind_name, iter_elapsed, made_progress, len(new_array_cols),
        )
        stage_times[f"flatten_loop_iter_{pass_index}"] = iter_elapsed

        if not made_progress:
            break

        if pass_index >= _FLATTEN_PASS_LIMIT:
            reached_flatten_pass_limit = True
            break

    if reached_flatten_pass_limit:
        logger.warning(
            "Reached flatten_pass_limit=%d for %s while changes were still being made.",
            _FLATTEN_PASS_LIMIT, kind_name,
        )

    t_stage = perf_counter()
    tags_result = extract_tags(df, kind_name)
    _record_stage("extract_tags", t_stage)
    if tags_result is not None:
        table_name, tags_df = tags_result
        children[table_name] = (_TAGS_COLUMN, tags_df)

    t_stage = perf_counter()
    if drop_wkt:
        drop_cols = [c for c in parent.columns if c.endswith("__wkt")]
        if drop_cols:
            parent = parent.drop(*drop_cols)
    _record_stage("drop_wkt", t_stage)

    t_stage = perf_counter()
    parent = _coerce_id_columns(parent)
    children = {name: (src_col, _coerce_id_columns(child_df)) for name, (src_col, child_df) in children.items()}
    _record_stage("coerce_id_columns", t_stage)

    t_stage = perf_counter()
    parent = _serialize_complex_columns(parent)
    _record_stage("serialize_complex_columns", t_stage)

    t_stage = perf_counter()
    if children and not globals().get("create_empty_child_tables", True):
        probe_frames: list[DataFrame] = []
        for name, (_, child_df) in children.items():
            probe_frames.append(child_df.select(F.lit(name).alias("__child_name")).limit(1))

        non_empty_names: set[str] = set()
        if probe_frames:
            probe_union = reduce(lambda left, right: left.unionByName(right), probe_frames)
            non_empty_names = {row["__child_name"] for row in probe_union.collect()}

        empty_children = [name for name in children if name not in non_empty_names]
        for name in empty_children:
            logger.info("Dropping empty child table %r (0 rows after explode)", name)
        children = {name: v for name, v in children.items() if name in non_empty_names}
    _record_stage("prune_empty_children", t_stage)

    logger.info(
        "Decomposed %s → parent (%d cols) + %d child tables",
        kind_name, len(parent.columns), len(children),
    )
    total_elapsed = perf_counter() - t_start
    logger.info("Decompose timing total for %s: %.3fs", kind_name, total_elapsed)
    for name, elapsed in sorted(stage_times.items(), key=lambda kv: kv[1], reverse=True):
        logger.info("  decompose stage %-32s %.3fs", name, elapsed)

    return DecomposedKind(
        kind_name=kind_name,
        col_classes=col_classes,
        parent=parent,
        children=children,
    )


# ── reassemble.py ─────────────────────────────────────────────────────



def _reassemble_key_columns(parent: DataFrame, child_df: DataFrame) -> list[str]:
    configured = globals().get("merge_key_columns", ["id", "version"])
    keys = [col for col in configured if col in parent.columns and col in child_df.columns]
    return keys or ["id"]


def _child_type(child_df: DataFrame) -> str:
    cols = set(child_df.columns)
    key_cols = set(_merge_key_column_names(child_df))
    value_cols = cols - key_cols
    if value_cols == {"tag_key", "tag_value"}:
        return "tags"
    if value_cols == {"ordinal", "value"}:
        return "primitive"
    return "struct"


def _child_suffix(table_name: str) -> str:
    if "___" in table_name:
        return table_name.split("___", 1)[1]
    return table_name


def _reassemble_tags(parent: DataFrame, child_df: DataFrame) -> DataFrame:
    key_cols = _reassemble_key_columns(parent, child_df)
    pivoted = child_df.groupBy(*key_cols).pivot("tag_key").agg(F.first("tag_value"))
    for col_name in pivoted.columns:
        if col_name not in key_cols:
            pivoted = pivoted.withColumnRenamed(col_name, f"tag_{col_name}")
    return parent.join(pivoted, on=key_cols, how="left")


def _reassemble_primitive(parent: DataFrame, child_df: DataFrame, suffix: str) -> DataFrame:
    key_cols = _reassemble_key_columns(parent, child_df)
    col_alias = suffix.replace("__", "_")
    agg_df = child_df.groupBy(*key_cols).agg(F.concat_ws(";", F.collect_list("value")).alias(col_alias))
    return parent.join(agg_df, on=key_cols, how="left")


def _reassemble_struct(
    parent: DataFrame,
    child_df: DataFrame,
    suffix: str,
    max_cardinality_cap: int,
) -> DataFrame:
    key_cols = _reassemble_key_columns(parent, child_df)
    max_card_row = child_df.groupBy(*key_cols).count().agg(F.max("count").alias("max_count")).collect()
    max_card = max_card_row[0]["max_count"] if max_card_row else 0
    if max_card is None:
        max_card = 0

    if max_card > max_cardinality_cap:
        logger.warning("Child '%s' has max cardinality %d, capping at %d", suffix, max_card, max_cardinality_cap)
        max_card = max_cardinality_cap

    value_fields = [c for c in child_df.columns if c not in (*key_cols, "ordinal")]

    for i in range(max_card):
        ordinal_slice = child_df.filter(F.col("ordinal") == i)
        select_exprs = [F.col(c) for c in key_cols]
        for fld in value_fields:
            alias = f"{suffix}_{i}_{fld}"
            select_exprs.append(F.col(fld).alias(alias))
        ordinal_df = ordinal_slice.select(select_exprs)
        parent = parent.join(ordinal_df, on=key_cols, how="left")

    return parent


def reassemble_kind(
    dk: DecomposedKind,
    max_cardinality_cap: int = 20,
) -> DataFrame:
    """Reassemble a DecomposedKind into a single flat DataFrame."""
    result = dk.parent

    for table_name, (source_col, child_df) in dk.children.items():
        suffix = _child_suffix(table_name)
        ctype = _child_type(child_df)

        if ctype == "tags":
            result = _reassemble_tags(result, child_df)
        elif ctype == "primitive":
            result = _reassemble_primitive(result, child_df, suffix)
        elif ctype == "struct":
            result = _reassemble_struct(result, child_df, suffix, max_cardinality_cap)

    return result


# ── Diagnostic schema-walk helpers ────────────────────────────────
# These helpers are not used by decompose_kind; keep them for optional schema inspection.


def collect_schema_fields(kind: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Walk the schema for ``kind``, returning (field_rows, ref_rows).

    Kept for diagnostic compatibility. The runtime decomposition path uses
    SchemaRegistry exclusively.
    """
    field_rows: list[dict[str, Any]] = []
    ref_rows: list[dict[str, Any]] = []
    visited: set[tuple[str, str]] = set()
    queue: deque[tuple[str, dict[str, Any], str]] = deque()

    root_doc, root_src = load_schema_doc(kind)
    root_name = root_doc.get("title") or parse_kind(kind)["entity_base"]
    queue.append((root_name, root_doc, root_src))

    def walk_properties(schema_name: str, schema_doc: dict[str, Any], path_prefix: str) -> None:
        properties = schema_doc.get("properties", {}) or {}
        for prop_name, spec in properties.items():
            if not isinstance(spec, dict):
                continue
            path = f"{path_prefix}.{prop_name}" if path_prefix else prop_name
            dtype = spec.get("type") or ("object" if "properties" in spec else "")
            field_rows.append(
                {
                    "schema": schema_name,
                    "field_path": path,
                    "type": dtype,
                    "resolved_type": dtype,
                }
            )
            if isinstance(spec.get("$ref"), str):
                nref = normalize_kind_ref(spec["$ref"])
                if nref:
                    ref_rows.append(
                        {
                            "from_schema": schema_name,
                            "field_path": path,
                            "normalized_ref": nref,
                            "resolved": False,
                            "source": "",
                        }
                    )
            if "allOf" in spec:
                for i, part in enumerate(spec["allOf"]):
                    if isinstance(part, dict) and part.get("$ref"):
                        nref = normalize_kind_ref(part["$ref"])
                        if nref:
                            ref_rows.append(
                                {
                                    "from_schema": schema_name,
                                    "field_path": f"{path}.allOf[{i}]",
                                    "normalized_ref": nref,
                                    "resolved": False,
                                    "source": "",
                                }
                            )
            if dtype == "object" and "properties" in spec:
                walk_properties(schema_name, spec, path)

    while queue:
        schema_name, schema_doc, schema_source = queue.popleft()
        schema_key = (schema_name, schema_source)
        if schema_key in visited:
            continue
        visited.add(schema_key)

        walk_properties(schema_name, schema_doc, "")

        for r in [x for x in ref_rows if x["from_schema"] == schema_name and not x["resolved"]]:
            try:
                child_doc, child_src = load_schema_doc(r["normalized_ref"])
                r["resolved"] = True
                r["source"] = child_src
                child_name = child_doc.get("title") or parse_kind(r["normalized_ref"])["entity_base"]
                queue.append((child_name, child_doc, child_src))
            except Exception:
                r["resolved"] = False

    return field_rows, ref_rows


print("Standalone Silver Layer decompose/reassemble loaded.")


## Pipeline functions

Load orchestration functions for processing one or more OSDU kinds.

- `process_kind()` reads bronze records, resolves schemas, transforms the data, and writes Silver Layer output for one kind.
- `run_silver_build()` runs `process_kind()` for all configured kinds and records status in `silver_run_info`.


In [ ]:
import re

_TABLE_EXISTS_CACHE: dict[str, bool] = {}


def _table_exists(spark: SparkSession, table_name: str, refresh: bool = False) -> bool:
    if not refresh and table_name in _TABLE_EXISTS_CACHE:
        return _TABLE_EXISTS_CACHE[table_name]
    try:
        if spark.catalog.tableExists(table_name):
            _TABLE_EXISTS_CACHE[table_name] = True
            return True
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise

    try:
        from delta.tables import DeltaTable

        exists = DeltaTable.isDeltaTable(spark, _table_path_uri(table_name))
        _TABLE_EXISTS_CACHE[table_name] = exists
        return exists
    except Exception as exc:
        logger.info("Delta table %r was not found through catalog or OneLake path: %s", table_name, exc)
        _TABLE_EXISTS_CACHE[table_name] = False
        return False


def _cast_null_columns(df: DataFrame) -> DataFrame:
    # Keep no-op for standalone compatibility
    return df


_TABLE_NAME_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def validate_table_names(table_names: list[str]) -> None:
    invalid = [name for name in table_names if not _TABLE_NAME_PATTERN.match(name or "")]
    if invalid:
        raise ValueError(
            "Invalid output table name(s): "
            + ", ".join(invalid)
            + ". Use letters, numbers, and underscores, and start with a letter or underscore."
        )


def _existing_tables(spark: SparkSession, table_names: list[str]) -> list[str]:
    return [name for name in table_names if _table_exists(spark, name, refresh=True)]


def _assert_overwrite_allowed(spark: SparkSession, table_names: list[str], allow_overwrite: bool) -> None:
    existing = _existing_tables(spark, table_names)
    if existing and not allow_overwrite:
        raise RuntimeError(
            "Full refresh would overwrite existing table(s): "
            + ", ".join(existing)
            + ". Set ALLOW_OVERWRITE = True or ADME_ALLOW_OVERWRITE=true to proceed."
        )


def _delta_table_for_target(spark: SparkSession, target: str):
    from delta.tables import DeltaTable

    try:
        return DeltaTable.forName(spark, target)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise

    return DeltaTable.forPath(spark, _table_path_uri(target))


def _incremental_child_write(
    spark: SparkSession,
    new_child_df: DataFrame,
    child_uri: str,
    changed_keys_df: DataFrame,
    merge_key_columns: list[str],
) -> None:
    """Replace child rows for changed parent merge keys, then append the new child rows."""
    _validate_merge_key_columns(new_child_df, merge_key_columns, f"Child DataFrame for {child_uri}")
    _validate_merge_key_columns(changed_keys_df, merge_key_columns, f"Changed keys for {child_uri}")
    if not _table_exists(spark, child_uri):
        logger.info("Creating new child Delta table for upsert target %s", child_uri)
        write_silver_table(new_child_df, child_uri, mode="overwrite")
        return

    changed_keys_df = changed_keys_df.select(*[F.col(col).alias(col) for col in merge_key_columns]).distinct()
    target = _delta_table_for_target(spark, child_uri)
    (
        target.alias("target")
        .merge(changed_keys_df.alias("changed"), _merge_condition("target", "changed", merge_key_columns))
        .whenMatchedDelete()
        .execute()
    )

    new_child_df = _cast_null_columns(new_child_df)
    _write_table(new_child_df, child_uri, mode="append")




# ── Incremental watermark state ──────────────────────────────────────


def _watermark_active(incremental: bool, watermark_column: str | None, watermark_mode: str | None) -> bool:
    return bool(incremental and watermark_column and (watermark_mode or "auto") != "off")


def _validate_watermark_column(df: DataFrame, watermark_column: str, context: str) -> T.DataType:
    if watermark_column not in df.columns:
        raise ValueError(f"{context} is missing configured watermark column {watermark_column!r}. Available columns: {df.columns}")
    return df.schema[watermark_column].dataType


def _watermark_state_table_name() -> str:
    return globals().get("incremental_state_table", "silver_incremental_state")


def load_incremental_watermark_state(spark: SparkSession, kinds: list[str], watermark_column: str) -> dict[str, str]:
    table_name = _watermark_state_table_name()
    if not watermark_column or not kinds or not _table_exists(spark, table_name):
        return {}
    latest_by_kind = Window.partitionBy("kind", "watermark_column").orderBy(F.col("updated_at").desc())
    try:
        rows = (
            _read_table_for_cache(table_name)
            .filter(F.col("kind").isin(list(dict.fromkeys(kinds))))
            .filter(F.col("watermark_column") == F.lit(watermark_column))
            .withColumn("__watermark_rank", F.row_number().over(latest_by_kind))
            .filter(F.col("__watermark_rank") == F.lit(1))
            .select("kind", "watermark_value")
            .collect()
        )
    except Exception as exc:
        raise RuntimeError(f"Could not read incremental watermark state from {table_name!r}.") from exc
    return {str(row["kind"]): str(row["watermark_value"]) for row in rows if row["watermark_value"] is not None}


def apply_incremental_watermark_filter(
    df: DataFrame,
    kind: str,
    watermark_column: str | None,
    watermark_mode: str | None,
    watermark_state: dict[str, str] | None,
) -> tuple[DataFrame, str | None, T.DataType | None]:
    mode = watermark_mode or "auto"
    if not watermark_column or mode == "off":
        return df, None, None
    if watermark_column not in df.columns:
        if mode == "required":
            _validate_watermark_column(df, watermark_column, f"Bronze rows for {kind}")
        logger.warning("Configured watermark column %r is not present for %s; processing all selected rows.", watermark_column, kind)
        return df, None, None
    data_type = df.schema[watermark_column].dataType
    previous = (watermark_state or {}).get(kind)
    if previous in (None, ""):
        return df, None, data_type
    return df.where(F.col(watermark_column).isNotNull() & (F.col(watermark_column) > F.lit(previous).cast(data_type))), previous, data_type


def max_watermark_value(df: DataFrame, watermark_column: str | None) -> tuple[str | None, str | None]:
    if not watermark_column:
        return None, None
    data_type = _validate_watermark_column(df, watermark_column, "Filtered bronze DataFrame")
    rows = df.agg(F.max(F.col(watermark_column)).cast("string").alias("watermark_value")).collect()
    if not rows:
        return None, data_type.simpleString()
    value = rows[0]["watermark_value"]
    return (str(value) if value is not None else None), data_type.simpleString()


def write_incremental_watermark_state(
    spark: SparkSession,
    rows: list[tuple[str, str, str, str | None, str | None]],
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    if not rows:
        return
    updated_at = datetime.now(UTC)
    state_rows = [(run_id, kind, column, value, data_type, updated_at) for run_id, kind, column, value, data_type in rows if value is not None]
    if not state_rows:
        return
    table_name = table_uri(workspace_id, lakehouse_id, _watermark_state_table_name())
    validate_table_names([table_name])
    df = spark.createDataFrame(state_rows, schema=INCREMENTAL_STATE_SCHEMA)
    _write_table(df, table_name, mode="append" if _table_exists(spark, table_name) else "overwrite")
    _TABLE_EXISTS_CACHE[table_name] = True


# ── Run-info metadata ───────────────────────────────────────────────


def write_run_info(
    spark: SparkSession,
    run_id: str,
    kind: str,
    start_time: datetime,
    end_time: datetime,
    records_processed: int,
    records_failed: int,
    status: str,
    error_message: str | None,
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    """Append a run_info record to the runtime-appropriate destination."""
    row = [
        (
            run_id,
            kind,
            start_time,
            end_time,
            records_processed,
            records_failed,
            status,
            error_message,
            (end_time - start_time).total_seconds() if end_time else None,
            None,
            globals().get("write_mode", "upsert" if globals().get("incremental", False) else "full_refresh"),
            globals().get("output_mode"),
            globals().get("merge_key_columns"),
            None,
            None,
            globals().get("watermark_column"),
            globals().get("watermark_mode"),
        )
    ]
    df = spark.createDataFrame(row, schema=RUN_INFO_SCHEMA)
    run_info_uri = table_uri(workspace_id, lakehouse_id, "silver_run_info")

    if _table_exists(spark, run_info_uri):
        _write_table(df, run_info_uri, mode="append")
    else:
        _write_table(df, run_info_uri, mode="overwrite")

    logger.info("Recorded run_info: %s %s → %s", run_id, kind, status)


def write_run_manifest(
    spark: SparkSession,
    run_id: str,
    result: KindResult,
    output_mode: str,
    table_prefix: str,
    bronze_table: str,
    run_profile: str,
    notebook_version: str,
    config_hash: str,
    allow_overwrite: bool,
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    """Append a run manifest row that describes output tables produced for one kind."""
    child_tables = result.child_tables or []
    row = [
        (
            run_id,
            result.kind,
            output_mode,
            globals().get("version_strategy", "merge"),
            kind_family_key(result.kind) if result.kind and ":" in result.kind else result.kind,
            [kind_version(result.kind)] if result.kind and ":" in result.kind else [],
            "missing" if result.status == "schema_missing" else "resolved",
            notebook_version,
            run_profile,
            config_hash,
            allow_overwrite,
            table_prefix,
            bronze_table,
            result.parent_table,
            child_tables,
            result.records_processed,
            result.status,
            result.error,
            datetime.now(UTC),
            globals().get("write_mode", "upsert" if globals().get("incremental", False) else "full_refresh"),
            globals().get("merge_key_columns"),
            len(child_tables),
            globals().get("output_docs_mode"),
            bool(globals().get("persist_schema_cache", False)),
            bool(globals().get("cache_bronze", False)),
            globals().get("watermark_column"),
            globals().get("watermark_mode"),
            bool(globals().get("include_inactive_records", False)),
        )
    ]
    df = spark.createDataFrame(row, schema=RUN_MANIFEST_SCHEMA)
    manifest_uri = table_uri(workspace_id, lakehouse_id, globals().get("run_manifest_table", "silver_run_manifest"))

    if _table_exists(spark, manifest_uri):
        _write_table(df, manifest_uri, mode="append")
    else:
        _write_table(df, manifest_uri, mode="overwrite")

    logger.info("Recorded run manifest: %s %s", run_id, result.kind)


def _output_doc_rows_for_df(
    run_id: str,
    kind: str,
    output_mode: str,
    table_name: str,
    table_role: str,
    source_column: str | None,
    df: DataFrame,
    notebook_version: str,
    config_hash: str,
    created_at: datetime,
) -> list[tuple]:
    rows: list[tuple] = []
    for ordinal, field in enumerate(df.schema.fields):
        rows.append(
            (
                run_id,
                kind,
                output_mode,
                table_name,
                table_role,
                source_column,
                field.name,
                field.dataType.simpleString(),
                ordinal,
                field.nullable,
                notebook_version,
                config_hash,
                created_at,
            )
        )
    return rows


def output_documentation_rows(
    run_id: str,
    kind: str,
    output_mode: str,
    parent_table: str,
    parent_df: DataFrame,
    children: dict[str, tuple[str, DataFrame]],
    notebook_version: str,
    config_hash: str,
) -> list[tuple]:
    docs_mode = globals().get("output_docs_mode", "summary")
    if not globals().get("write_output_docs", True) or docs_mode == "off":
        return []

    docs_table = globals().get("output_docs_table", "silver_output_documentation")
    validate_table_names([docs_table, parent_table, *children.keys()])
    created_at = datetime.now(UTC)
    if docs_mode == "summary":
        rows = [
            (
                run_id,
                kind,
                output_mode,
                parent_table,
                "wide" if output_mode == "wide" else "parent",
                None,
                "*",
                f"{len(parent_df.columns)} columns",
                0,
                None,
                notebook_version,
                config_hash,
                created_at,
            )
        ]
        if output_mode == "normalized":
            for child_table, (source_col, child_df) in children.items():
                rows.append(
                    (
                        run_id,
                        kind,
                        output_mode,
                        child_table,
                        "child",
                        source_col,
                        "*",
                        f"{len(child_df.columns)} columns",
                        0,
                        None,
                        notebook_version,
                        config_hash,
                        created_at,
                    )
                )
    else:
        rows = _output_doc_rows_for_df(
            run_id,
            kind,
            output_mode,
            parent_table,
            "wide" if output_mode == "wide" else "parent",
            None,
            parent_df,
            notebook_version,
            config_hash,
            created_at,
        )
        if output_mode == "normalized":
            for child_table, (source_col, child_df) in children.items():
                rows.extend(
                    _output_doc_rows_for_df(
                        run_id,
                        kind,
                        output_mode,
                        child_table,
                        "child",
                        source_col,
                        child_df,
                        notebook_version,
                        config_hash,
                        created_at,
                    )
                )
    return rows


def flush_output_documentation_rows(
    spark: SparkSession,
    output_docs_rows: list[tuple],
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    if not output_docs_rows:
        return

    docs_table = globals().get("output_docs_table", "silver_output_documentation")
    validate_table_names([docs_table])
    df = spark.createDataFrame(list(output_docs_rows), schema=OUTPUT_DOCS_SCHEMA)
    docs_uri = table_uri(workspace_id, lakehouse_id, docs_table)
    mode = "append" if _table_exists(spark, docs_uri) else "overwrite"
    _write_table(df, docs_uri, mode=mode)
    _TABLE_EXISTS_CACHE[docs_uri] = True
    output_docs_rows.clear()


def write_output_documentation(
    spark: SparkSession,
    run_id: str,
    kind: str,
    output_mode: str,
    parent_table: str,
    parent_df: DataFrame,
    children: dict[str, tuple[str, DataFrame]],
    workspace_id: str,
    lakehouse_id: str,
    notebook_version: str,
    config_hash: str,
) -> None:
    """Append generated table/column documentation for outputs produced by one kind."""
    rows = output_documentation_rows(
        run_id,
        kind,
        output_mode,
        parent_table,
        parent_df,
        children,
        notebook_version,
        config_hash,
    )
    has_rows = bool(rows)
    flush_output_documentation_rows(spark, rows, workspace_id, lakehouse_id)
    if has_rows:
        logger.info("Recorded output documentation for %s in %s", kind, globals().get("output_docs_table", "silver_output_documentation"))


def _buffer_or_write_output_documentation(
    spark: SparkSession,
    run_id: str,
    kind: str,
    output_mode: str,
    parent_table: str,
    parent_df: DataFrame,
    children: dict[str, tuple[str, DataFrame]],
    workspace_id: str,
    lakehouse_id: str,
    notebook_version: str,
    config_hash: str,
    output_docs_rows: list[tuple] | None,
) -> None:
    rows = output_documentation_rows(
        run_id,
        kind,
        output_mode,
        parent_table,
        parent_df,
        children,
        notebook_version,
        config_hash,
    )
    if output_docs_rows is None:
        flush_output_documentation_rows(spark, rows, workspace_id, lakehouse_id)
    else:
        output_docs_rows.extend(rows)


def process_kind(
    spark: SparkSession,
    kind: str,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    limit: int | None = None,
    incremental: bool = False,
    drop_wkt: bool = True,
    reassemble: bool = False,
    table_prefix: str = "",
    allow_overwrite: bool = False,
    run_id: str = "",
    notebook_version: str = "",
    config_hash: str = "",
) -> KindResult:
    """Process one OSDU kind from bronze to silver tables.

    Uses the standalone verbatim port of decompose/reassemble with a SchemaRegistry
    built from the ADME schema service (substitute for SchemaRegistry.from_spark).
    """
    kind_name = kind_to_table_name(kind)
    parent_table = f"{table_prefix}{kind_name}"
    validate_table_names([parent_table])

    bronze_df = read_bronze_kind_spark(
        kind,
        spark,
        workspace_id,
        lakehouse_id,
        bronze_table=bronze_table,
    )
    if limit:
        bronze_df = bronze_df.limit(limit)

    records_processed = bronze_df.count()
    if records_processed == 0:
        return KindResult(
            kind=kind,
            status="skipped",
            records_processed=0,
            parent_table=parent_table,
            child_tables=[],
            reassembled=reassemble,
            validation_passed=True,
        )

    # Build SchemaRegistry from the ADME schema service (standalone substitute
    # for SchemaRegistry.from_spark used in the packaged build).
    registry = build_registry_from_adme([kind])
    if not registry.has_kind(kind):
        detail = _schema_load_error(kind) if "_schema_load_error" in globals() else None
        message = f"Schema for {kind!r} could not be loaded from ADME schema service; aborting."
        if detail:
            message = f"{message} {detail}"
        raise RuntimeError(message)

    decomposed = decompose_kind(
        bronze_df,
        kind_name=parent_table,
        drop_wkt=drop_wkt,
        registry=registry,
        osdu_kind=kind,
        input_rows=records_processed,
        infer_nested_json=True,
    )

    parent_df = decomposed.parent
    children = decomposed.children

    # Fail fast if JSON-heavy payload was not flattened.
    if "data" in bronze_df.columns:
        has_data_flattened = any(c.startswith("data__") for c in parent_df.columns)
        if not has_data_flattened:
            raise RuntimeError(
                "Decomposition did not flatten 'data' into parent columns. "
                "Rerun implementation cells before pipeline execution."
            )

    output_tables = [parent_table] if reassemble else [parent_table, *children.keys()]
    validate_table_names(output_tables)
    if not incremental:
        _assert_overwrite_allowed(spark, output_tables, allow_overwrite)

    if reassemble:
        flat_df = reassemble_kind(decomposed)
        if incremental:
            upsert_silver_table(flat_df, parent_table, merge_key_columns=merge_keys)
        else:
            write_silver_table(flat_df, parent_table, mode="overwrite")
        write_output_documentation(
            spark,
            run_id,
            kind,
            "wide",
            parent_table,
            flat_df,
            {},
            workspace_id,
            lakehouse_id,
            notebook_version,
            config_hash,
        )
        return KindResult(
            kind=kind,
            status="success",
            records_processed=records_processed,
            records_failed=0,
            parent_table=parent_table,
            child_tables=[],
            reassembled=True,
            validation_passed=True,
        )

    # Parent + children mode
    if incremental:
        upsert_silver_table(parent_df, parent_table)
        changed_ids_df = parent_df.select(F.col("id").cast("string").alias("id")).distinct()
        for child_table, (_, child_df) in children.items():
            _incremental_child_write(
                spark,
                child_df,
                child_table,
                changed_ids_df,
            )
    else:
        write_silver_table(parent_df, parent_table, mode="overwrite")
        for child_table, (_, child_df) in children.items():
            write_silver_table(child_df, child_table, mode="overwrite")

    write_output_documentation(
        spark,
        run_id,
        kind,
        "normalized",
        parent_table,
        parent_df,
        children,
        workspace_id,
        lakehouse_id,
        notebook_version,
        config_hash,
    )

    return KindResult(
        kind=kind,
        status="success",
        records_processed=records_processed,
        records_failed=0,
        parent_table=parent_table,
        child_tables=list(children.keys()),
        reassembled=False,
        validation_passed=True,
    )

def _with_schema_metadata(df: DataFrame, kind: str) -> DataFrame:
    p = parse_kind(kind)
    return (
        df.withColumn("schema_authority", F.lit(p["authority"]))
        .withColumn("schema_source", F.lit(p["source"]))
        .withColumn("schema_entity", F.lit(p["entity"]))
        .withColumn("schema_version", F.lit(p["version"]))
        .withColumn("osdu_kind", F.lit(kind))
    )


def _union_frames(frames: list[DataFrame]) -> DataFrame:
    if not frames:
        raise ValueError("No DataFrames to union")
    result = frames[0]
    for frame in frames[1:]:
        result = result.unionByName(frame, allowMissingColumns=True)
    return result


def _schema_missing_result(kind: str, parent_table: str, records_processed: int, message: str, reassemble: bool) -> KindResult:
    return KindResult(
        kind=kind,
        status="schema_missing",
        records_processed=records_processed,
        records_failed=records_processed,
        parent_table=parent_table,
        child_tables=[],
        reassembled=reassemble,
        validation_passed=False,
        error=message,
    )


def process_kind_group(
    spark: SparkSession,
    group: dict[str, object],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    limit: int | None = None,
    kind_limits: dict[str, int] | None = None,
    incremental: bool = False,
    drop_wkt: bool = True,
    reassemble: bool = False,
    table_prefix: str = "",
    allow_overwrite: bool = False,
    run_id: str = "",
    notebook_version: str = "",
    config_hash: str = "",
    missing_schema_mode: str = "skip",
    bronze_df: DataFrame | None = None,
    kind_counts: dict[str, int] | None = None,
    schema_registry: SchemaRegistry | None = None,
    schema_status: dict[str, str] | None = None,
    merge_key_columns: list[str] | None = None,
    output_docs_rows: list[tuple] | None = None,
    watermark_column: str | None = None,
    watermark_mode: str = "auto",
    watermark_state: dict[str, str] | None = None,
) -> list[KindResult]:
    """Process one version group and write each logical output table once."""
    merge_keys = _effective_merge_key_columns(merge_key_columns)
    group_kinds = list(group["kinds"])
    parent_table = table_name_for_kind_group(group, table_prefix)
    validate_table_names([parent_table])

    parent_frames: list[DataFrame] = []
    flat_frames: list[DataFrame] = []
    child_frames: dict[str, list[DataFrame]] = {}
    child_sources: dict[str, str] = {}
    results: list[KindResult] = []
    successful_kinds: list[str] = []
    watermark_updates: list[tuple[str, str, str, str | None, str | None]] = []

    for kind in group_kinds:
        effective_limit = limit_for_kind(kind, limit, kind_limits)
        kind_bronze_df = read_bronze_kind_spark(
            kind,
            spark,
            workspace_id,
            lakehouse_id,
            bronze_table=bronze_table,
            bronze_df=bronze_df,
        )
        previous_watermark = None
        watermark_data_type = None
        if _watermark_active(incremental, watermark_column, watermark_mode):
            kind_bronze_df, previous_watermark, watermark_data_type = apply_incremental_watermark_filter(
                kind_bronze_df,
                kind,
                watermark_column,
                watermark_mode,
                watermark_state,
            )
        if effective_limit:
            kind_bronze_df = kind_bronze_df.limit(effective_limit)

        known_count = None if _watermark_active(incremental, watermark_column, watermark_mode) else (kind_counts.get(kind) if kind_counts else None)
        records_processed = min(known_count, effective_limit) if known_count is not None and effective_limit else known_count
        if records_processed is None:
            records_processed = kind_bronze_df.count()
        if records_processed == 0:
            results.append(
                KindResult(
                    kind=kind,
                    status="skipped",
                    records_processed=0,
                    parent_table=parent_table,
                    child_tables=[],
                    reassembled=reassemble,
                    validation_passed=True,
                )
            )
            continue

        registry = schema_registry or build_registry_from_adme([kind])
        status = (schema_status or {}).get(kind)
        if status == "missing" or not registry.has_kind(kind):
            message = f"Schema for {kind!r} could not be resolved."
            detail = _schema_load_error(kind) if "_schema_load_error" in globals() else None
            if detail:
                message = f"{message} {detail}"
            if missing_schema_mode == "fail":
                raise RuntimeError(message)
            if missing_schema_mode == "infer":
                message += " Missing-schema inference is best-effort and is not enabled in this schema-correct path; skipping."
            results.append(_schema_missing_result(kind, parent_table, records_processed, message, reassemble))
            continue

        new_watermark_value, new_watermark_type = max_watermark_value(kind_bronze_df, watermark_column if _watermark_active(incremental, watermark_column, watermark_mode) else None)

        decomposed = decompose_kind(
            kind_bronze_df,
            kind_name=parent_table,
            drop_wkt=drop_wkt,
            registry=registry,
            osdu_kind=kind,
            input_rows=records_processed,
            infer_nested_json=True,
        )

        if reassemble:
            flat_df = _with_schema_metadata(reassemble_kind(decomposed), kind)
            flat_frames.append(flat_df)
        else:
            parent_frames.append(_with_schema_metadata(decomposed.parent, kind))
            for child_table, (source_col, child_df) in decomposed.children.items():
                child_frames.setdefault(child_table, []).append(_with_schema_metadata(child_df, kind))
                child_sources.setdefault(child_table, source_col)

        successful_kinds.append(kind)
        if _watermark_active(incremental, watermark_column, watermark_mode) and new_watermark_value is not None:
            watermark_updates.append((run_id, kind, watermark_column or "", new_watermark_value, new_watermark_type))
        results.append(
            KindResult(
                kind=kind,
                status="success",
                records_processed=records_processed,
                records_failed=0,
                parent_table=parent_table,
                child_tables=[],
                reassembled=reassemble,
                validation_passed=True,
            )
        )

    if not successful_kinds:
        return results

    if reassemble:
        flat_df = _union_frames(flat_frames)
        output_tables = [parent_table]
        validate_table_names(output_tables)
        if not incremental:
            _assert_overwrite_allowed(spark, output_tables, allow_overwrite)
        if incremental:
            upsert_silver_table(flat_df, parent_table, merge_key_columns=merge_keys)
        else:
            write_silver_table(flat_df, parent_table, mode="overwrite")
        _buffer_or_write_output_documentation(
            spark,
            run_id,
            str(group["group_key"]),
            "wide",
            parent_table,
            flat_df,
            {},
            workspace_id,
            lakehouse_id,
            notebook_version,
            config_hash,
            output_docs_rows,
        )
        for result in results:
            if result.status == "success":
                result.reassembled = True
                result.child_tables = []
        write_incremental_watermark_state(spark, watermark_updates, workspace_id, lakehouse_id)
        return results

    parent_df = _union_frames(parent_frames)
    unioned_children: dict[str, tuple[str, DataFrame]] = {
        child_table: (child_sources.get(child_table, ""), _union_frames(frames))
        for child_table, frames in child_frames.items()
    }
    child_tables = list(unioned_children.keys())
    output_tables = [parent_table, *child_tables]
    validate_table_names(output_tables)
    if not incremental:
        _assert_overwrite_allowed(spark, output_tables, allow_overwrite)

    if incremental:
        upsert_silver_table(parent_df, parent_table, merge_key_columns=merge_keys)
        changed_keys_df = parent_df.select(*[F.col(col).alias(col) for col in merge_keys]).distinct()
        for child_table, (_, child_df) in unioned_children.items():
            _incremental_child_write(spark, child_df, child_table, changed_keys_df, merge_keys)
    else:
        write_silver_table(parent_df, parent_table, mode="overwrite")
        for child_table, (_, child_df) in unioned_children.items():
            write_silver_table(child_df, child_table, mode="overwrite")

    _buffer_or_write_output_documentation(
        spark,
        run_id,
        str(group["group_key"]),
        "normalized",
        parent_table,
        parent_df,
        unioned_children,
        workspace_id,
        lakehouse_id,
        notebook_version,
        config_hash,
        output_docs_rows,
    )

    for result in results:
        if result.status == "success":
            result.child_tables = child_tables
            result.reassembled = False
    write_incremental_watermark_state(spark, watermark_updates, workspace_id, lakehouse_id)
    return results



In [ ]:
from time import perf_counter

def kind_parts(kind: str) -> dict[str, str]:
    authority, source, entity_ver = kind.split(":", 2)
    entity, version = entity_ver.rsplit(":", 1)
    return {
        "authority": authority,
        "source": source,
        "entity": entity,
        "entity_base": entity.split("--")[-1],
        "version": version,
    }


def kind_family_key(kind: str) -> str:
    p = kind_parts(kind)
    return f"{p['authority']}:{p['source']}:{p['entity']}"


def kind_version(kind: str) -> str:
    return kind_parts(kind)["version"]


def kind_to_versioned_table_name(kind: str) -> str:
    base = kind_to_table_name(kind)
    version = kind_version(kind).replace(".", "_").replace("-", "_")
    return f"{base}__v{version}"


def group_kinds_by_version_strategy(kinds: list[str], version_strategy: str) -> list[dict[str, object]]:
    if version_strategy == "versioned_tables":
        return [
            {
                "group_key": f"{kind_family_key(kind)}:{kind_version(kind)}",
                "kinds": [kind],
                "parent_table_base": kind_to_versioned_table_name(kind),
                "versions": [kind_version(kind)],
            }
            for kind in kinds
        ]

    groups: dict[str, dict[str, object]] = {}
    for kind in kinds:
        key = kind_family_key(kind)
        group = groups.setdefault(
            key,
            {
                "group_key": key,
                "kinds": [],
                "parent_table_base": kind_to_table_name(kind),
                "versions": [],
            },
        )
        group["kinds"].append(kind)
        group["versions"].append(kind_version(kind))

    for group in groups.values():
        group["kinds"] = sorted(group["kinds"], key=kind_version)
        group["versions"] = sorted(set(group["versions"]))
    return list(groups.values())


def table_name_for_kind_group(group: dict[str, object], table_prefix: str) -> str:
    return f"{table_prefix}{group['parent_table_base']}"


def detect_table_collisions(kinds: list[str], table_prefix: str, version_strategy: str) -> list[dict[str, object]]:
    rows: dict[str, list[str]] = {}
    for group in group_kinds_by_version_strategy(kinds, version_strategy):
        table_name = table_name_for_kind_group(group, table_prefix)
        rows.setdefault(table_name, []).extend(group["kinds"])
    return [
        {"table_name": table_name, "kinds": table_kinds, "safe": version_strategy == "merge" and len({kind_family_key(k) for k in table_kinds}) == 1}
        for table_name, table_kinds in rows.items()
        if len(table_kinds) > 1
    ]



def compute_kind_counts(bronze_df: DataFrame, kinds: list[str] | None = None) -> dict[str, int]:
    df = bronze_df
    if kinds:
        df = df.where(F.col("kind").isin(kinds))
    return {row["kind"]: int(row["count"]) for row in df.groupBy("kind").count().collect()}


def prefetch_schema_registry(kinds: list[str], enabled: bool = True) -> tuple[SchemaRegistry | None, dict[str, str]]:
    if not enabled:
        return None, {}
    try:
        registry = build_registry_from_adme(kinds)
        status = {kind: ("resolved" if registry.has_kind(kind) else "missing") for kind in kinds}
        return registry, status
    except Exception as exc:
        logger.warning("Schema preflight failed; falling back to per-kind schema resolution: %s", exc)
        return None, {}


def _metadata_mode(table_name: str, spark: SparkSession) -> str:
    return "append" if _table_exists(spark, table_name) else "overwrite"


def _timings_json(timings: dict[str, float]) -> str:
    return json.dumps({name: round(float(elapsed), 3) for name, elapsed in sorted(timings.items())}, sort_keys=True)


def flush_metadata_buffers(
    spark: SparkSession,
    run_info_rows: list[tuple],
    manifest_rows: list[tuple],
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    if run_info_rows:
        table_name = table_uri(workspace_id, lakehouse_id, "silver_run_info")
        df = spark.createDataFrame(list(run_info_rows), schema=RUN_INFO_SCHEMA)
        _write_table(df, table_name, mode=_metadata_mode(table_name, spark))
        _TABLE_EXISTS_CACHE[table_name] = True
        run_info_rows.clear()
    if manifest_rows:
        table_name = table_uri(workspace_id, lakehouse_id, globals().get("run_manifest_table", "silver_run_manifest"))
        df = spark.createDataFrame(list(manifest_rows), schema=RUN_MANIFEST_SCHEMA)
        _write_table(df, table_name, mode=_metadata_mode(table_name, spark))
        _TABLE_EXISTS_CACHE[table_name] = True
        manifest_rows.clear()


def run_info_row(
    run_id: str,
    kind: str,
    start_time: datetime,
    end_time: datetime,
    result: KindResult,
    error_message: str | None,
    error_type: str | None,
    write_mode: str,
    output_mode: str,
    merge_key_columns: list[str],
    schema_access_detail: str | None,
    stage_timings: dict[str, float],
    watermark_column: str | None,
    watermark_mode: str | None,
) -> tuple:
    return (
        run_id,
        kind,
        start_time,
        end_time,
        result.records_processed,
        result.records_failed,
        result.status,
        error_message,
        (end_time - start_time).total_seconds() if end_time else None,
        error_type,
        write_mode,
        output_mode,
        merge_key_columns,
        schema_access_detail,
        _timings_json(stage_timings),
        watermark_column,
        watermark_mode,
    )


def run_manifest_row(
    run_id: str,
    result: KindResult,
    output_mode: str,
    table_prefix: str,
    bronze_table: str,
    run_profile: str,
    notebook_version: str,
    config_hash: str,
    allow_overwrite: bool,
    write_mode: str,
    merge_key_columns: list[str],
    watermark_column: str | None,
    watermark_mode: str | None,
) -> tuple:
    child_tables = result.child_tables or []
    return (
        run_id,
        result.kind,
        output_mode,
        globals().get("version_strategy", "merge"),
        kind_family_key(result.kind) if result.kind and ":" in result.kind else result.kind,
        [kind_version(result.kind)] if result.kind and ":" in result.kind else [],
        "missing" if result.status == "schema_missing" else "resolved",
        notebook_version,
        run_profile,
        config_hash,
        allow_overwrite,
        table_prefix,
        bronze_table,
        result.parent_table,
        child_tables,
        result.records_processed,
        result.status,
        result.error,
        datetime.now(UTC),
        write_mode,
        merge_key_columns,
        len(child_tables),
        globals().get("output_docs_mode"),
        bool(globals().get("persist_schema_cache", False)),
        bool(globals().get("cache_bronze", False)),
        watermark_column,
        watermark_mode,
        bool(globals().get("include_inactive_records", False)),
    )

def is_all_kinds_selector(value: str | None) -> bool:
    selector = (value or "").strip()
    return selector == "*" or selector == "*:*:*:*" or selector.lower() == "all"


def is_kind_pattern(value: str | None) -> bool:
    selector = (value or "").strip()
    return is_all_kinds_selector(selector) or "*" in selector


def kind_pattern_to_regex(pattern: str):
    import re

    escaped = re.escape(pattern.strip())
    return re.compile("^" + escaped.replace(r"\*", ".*") + "$")


def matches_kind_selector(kind: str, selector: str) -> bool:
    selector = selector.strip()
    if is_all_kinds_selector(selector):
        return True
    if "*" in selector:
        return bool(kind_pattern_to_regex(selector).match(kind))
    return kind == selector


def _clean_kind_selectors(selectors: list[str]) -> list[str]:
    cleaned = [str(selector).strip() for selector in selectors if str(selector).strip()]
    if not cleaned:
        raise ValueError("At least one OSDU kind or wildcard selector is required.")
    return list(dict.fromkeys(cleaned))


def kind_selectors_require_discovery(selectors: list[str]) -> bool:
    return any(is_kind_pattern(selector) for selector in selectors)


def discover_bronze_kinds(
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    bronze_df: DataFrame | None = None,
) -> list[str]:
    df = bronze_df if bronze_df is not None else read_bronze_table_spark(spark, workspace_id, lakehouse_id, bronze_table=bronze_table)
    if "kind" not in df.columns:
        raise ValueError(f"Bronze table '{bronze_table}' does not contain a 'kind' column for wildcard discovery.")
    return sorted(row["kind"] for row in df.select("kind").where(F.col("kind").isNotNull()).distinct().collect())


def resolve_kind_selectors(
    selectors: list[str],
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    bronze_df: DataFrame | None = None,
) -> list[str]:
    cleaned = _clean_kind_selectors(selectors)
    if not kind_selectors_require_discovery(cleaned):
        return cleaned

    discovered = discover_bronze_kinds(spark, workspace_id, lakehouse_id, bronze_table, bronze_df=bronze_df)
    if not discovered:
        raise ValueError(f"No distinct kind values were found in bronze table '{bronze_table}'.")

    resolved: list[str] = []
    unmatched_patterns: list[str] = []
    for selector in cleaned:
        if is_kind_pattern(selector):
            matches = [kind for kind in discovered if matches_kind_selector(kind, selector)]
            if not matches:
                unmatched_patterns.append(selector)
            resolved.extend(matches)
        else:
            resolved.append(selector)

    if unmatched_patterns:
        raise ValueError("Wildcard kind selector(s) matched no bronze kinds: " + ", ".join(unmatched_patterns))

    resolved = list(dict.fromkeys(resolved))
    if not resolved:
        raise ValueError("Kind selectors resolved to no concrete kinds.")
    return resolved


def ensure_resolved_kinds(
    selectors: list[str],
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    bronze_df: DataFrame | None = None,
) -> list[str]:
    resolved = resolve_kind_selectors(selectors, spark, workspace_id, lakehouse_id, bronze_table, bronze_df=bronze_df)
    globals()["resolved_kinds"] = resolved
    globals()["kinds"] = resolved
    return resolved


def _kind_limit_candidates(kind: str) -> list[str]:
    table_name = kind_to_table_name(kind)
    entity = kind.rsplit(":", 1)[0].split("--")[-1] if "--" in kind else kind
    return [kind, table_name, entity]


def limit_for_kind(kind: str, default_limit: int | None, kind_limits: dict[str, int] | None) -> int | None:
    for key in _kind_limit_candidates(kind):
        if kind_limits and key in kind_limits:
            value = kind_limits[key]
            return value or None
    return default_limit


def _check_row(name: str, passed: bool, detail: str) -> dict[str, str | bool]:
    icon = "✓" if passed else "✗"
    print(f"  {icon} {name}: {detail}")
    return {"check": name, "passed": passed, "detail": detail}


def preview_output_tables(
    kinds: list[str],
    table_prefix: str,
    output_mode: str,
    limit: int | None = None,
    kind_limits: dict[str, int] | None = None,
    version_strategy: str | None = None,
) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    strategy = version_strategy or globals().get("version_strategy", "merge")
    for group in group_kinds_by_version_strategy(kinds, strategy):
        parent_table = table_name_for_kind_group(group, table_prefix)
        validate_table_names([parent_table])
        limits = [str(limit_for_kind(kind, limit, kind_limits) or "none") for kind in group["kinds"]]
        rows.append(
            {
                "kind": ", ".join(group["kinds"][:3]) + (" ..." if len(group["kinds"]) > 3 else ""),
                "kind_count": str(len(group["kinds"])),
                "versions": ",".join(group["versions"]),
                "output_mode": output_mode,
                "parent_table": parent_table,
                "effective_limit": ",".join(limits),
                "child_tables": "discovered during decomposition" if output_mode == "normalized" else "not created in wide mode",
            }
        )
    return rows


def run_setup_checklist(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    output_mode: str,
    table_prefix: str,
    limit: int | None = None,
    kind_limits: dict[str, int] | None = None,
    incremental: bool = False,
    watermark_column: str | None = None,
    watermark_mode: str = "auto",
) -> list[dict[str, str | bool]]:
    """Validate configuration and read access before full execution."""
    print("Setup checklist")
    checks: list[dict[str, str | bool]] = []
    selectors = _clean_kind_selectors(kinds)

    checks.append(_check_row("workspace", bool(workspace_id), workspace_id or "missing"))
    checks.append(_check_row("lakehouse", bool(lakehouse_id), lakehouse_id or "missing"))
    checks.append(_check_row("bronze table", bool(bronze_table), bronze_table or "missing"))
    checks.append(_check_row("kind selectors", bool(selectors), f"{len(selectors)} selector(s): {selectors[:5]}"))
    checks.append(_check_row("output mode", output_mode in {"normalized", "wide"}, output_mode))

    try:
        raw_bronze_df = read_bronze_table_spark(spark, workspace_id, lakehouse_id, bronze_table=bronze_table, apply_active_filter=False)
        passed, detail = active_record_filter_status(raw_bronze_df)
        checks.append(_check_row("active record filter", passed, detail))
    except Exception as exc:
        checks.append(_check_row("active record filter", False, str(exc)))

    try:
        checks.append(_check_row("ADME schema access", True, validate_adme_schema_service_access()))
    except Exception as exc:
        checks.append(_check_row("ADME schema access", False, str(exc)))

    try:
        kinds = ensure_resolved_kinds(selectors, spark, workspace_id, lakehouse_id, bronze_table)
        preview = kinds[:5]
        detail = f"{len(kinds)} resolved"
        if preview:
            detail += f"; preview={preview}"
        checks.append(_check_row("resolved kinds", True, detail))
    except Exception as exc:
        checks.append(_check_row("resolved kinds", False, str(exc)))
        kinds = []

    if kinds:
        try:
            sample_df = read_bronze_kind_spark(kinds[0], spark, workspace_id, lakehouse_id, bronze_table=bronze_table, limit=1)
            sample_rows = sample_df.take(1)
            checks.append(_check_row("bronze access", True, f"read {len(sample_rows)} preview row(s) for first kind"))
            if incremental and watermark_column and watermark_mode != "off":
                if watermark_column in sample_df.columns:
                    checks.append(_check_row("watermark column", True, f"{watermark_column} is available for source-change filtering"))
                elif watermark_mode == "required":
                    checks.append(_check_row("watermark column", False, f"required column {watermark_column!r} is missing"))
                else:
                    checks.append(_check_row("watermark column", True, f"{watermark_column!r} missing; upsert mode will process all selected rows"))
            elif incremental:
                checks.append(_check_row("watermark column", True, "not configured; upsert mode will process all selected rows"))
        except Exception as exc:
            checks.append(_check_row("bronze access", False, str(exc)))

        try:
            registry = build_registry_from_adme([kinds[0]])
            schema_ok = registry.has_kind(kinds[0])
            detail = f"resolved {kinds[0]}" if schema_ok else (_schema_load_error(kinds[0]) if "_schema_load_error" in globals() else f"missing {kinds[0]}")
            checks.append(_check_row("first-kind schema", schema_ok, detail))
        except Exception as exc:
            checks.append(_check_row("first-kind schema", False, str(exc)))

    planned = preview_output_tables(kinds, table_prefix, output_mode, limit=limit, kind_limits=kind_limits, version_strategy=globals().get("version_strategy", "merge"))
    collisions = detect_table_collisions(kinds, table_prefix, globals().get("version_strategy", "merge"))
    unsafe_collisions = [row for row in collisions if not row["safe"]]
    checks.append(_check_row("version groups", True, f"{len(planned)} planned table group(s); {len(collisions)} multi-kind group(s)"))
    if unsafe_collisions:
        checks.append(_check_row("table collisions", False, str(unsafe_collisions[:3])))
    else:
        checks.append(_check_row("table collisions", True, "none unsafe"))
    metadata_tables = [
        globals().get("schema_cache_table", "silver_schema_cache"),
        globals().get("run_manifest_table", "silver_run_manifest"),
        globals().get("output_docs_table", "silver_output_documentation"),
        "silver_run_info",
    ]
    try:
        validate_table_names([row["parent_table"] for row in planned] + metadata_tables)
        checks.append(_check_row("table names", True, "all planned table names are valid"))
    except Exception as exc:
        checks.append(_check_row("table names", False, str(exc)))

    print("Planned output tables:")
    for row in planned:
        print(f"  - {row['parent_table']} ({row['output_mode']}, kinds={row['kind_count']}, versions={row['versions']}, limit={row['effective_limit']})")

    passed = all(bool(row["passed"]) for row in checks)
    print(f"Setup checklist {'passed' if passed else 'failed'}")
    return checks


def run_silver_dry_run(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    limit: int | None,
    output_mode: str,
    table_prefix: str,
    kind_limits: dict[str, int] | None = None,
) -> list[dict[str, str | int | bool]]:
    """Validate inputs and preview planned writes without creating Silver Layer tables."""
    print("Dry run: validating configuration and previewing planned writes")
    print(f"  Include inactive records: {globals().get('include_inactive_records', False)}")
    checks = run_setup_checklist(
        spark,
        kinds,
        workspace_id,
        lakehouse_id,
        bronze_table,
        output_mode,
        table_prefix,
        limit=limit,
        kind_limits=kind_limits,
        incremental=globals().get("incremental", False),
        watermark_column=globals().get("watermark_column"),
        watermark_mode=globals().get("watermark_mode", "auto"),
    )
    kinds = globals().get("resolved_kinds", kinds)
    dry_rows: list[dict[str, str | int | bool]] = []

    for kind in kinds:
        parent_table = f"{table_prefix}{kind_to_table_name(kind)}"
        effective_limit = limit_for_kind(kind, limit, kind_limits)
        row: dict[str, str | int | bool] = {
            "kind": kind,
            "output_mode": output_mode,
            "parent_table": parent_table,
            "effective_limit": effective_limit or "none",
            "schema_resolved": False,
            "preview_rows": 0,
            "would_write": False,
        }
        try:
            registry = build_registry_from_adme([kind])
            row["schema_resolved"] = registry.has_kind(kind)
            if not row["schema_resolved"] and "_schema_load_error" in globals():
                row["error"] = _schema_load_error(kind) or "schema not found"
            preview_df = read_bronze_kind_spark(kind, spark, workspace_id, lakehouse_id, bronze_table=bronze_table, limit=effective_limit or 1)
            row["preview_rows"] = len(preview_df.take(1))
            row["would_write"] = bool(row["schema_resolved"])
        except Exception as exc:
            row["error"] = str(exc)
        dry_rows.append(row)

    print("Dry run planned writes:")
    for row in dry_rows:
        print(f"  - {row['kind']} → {row['parent_table']} ({row['output_mode']}) limit={row['effective_limit']} schema={row['schema_resolved']} preview_rows={row['preview_rows']}")
    print("Dry run complete: no Silver Layer tables were written.")
    return dry_rows


def run_silver_build(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str | None = None,
    limit: int | None = None,
    incremental: bool = False,
    drop_wkt: bool = True,
    reassemble: bool = False,
    table_prefix: str = "",
    allow_overwrite: bool = False,
    run_profile: str = "full",
    notebook_version: str = "",
    config_hash: str = "",
    kind_limits: dict[str, int] | None = None,
    version_strategy: str = "merge",
    missing_schema_mode: str = "skip",
    merge_key_columns: list[str] | None = None,
) -> list[KindResult]:
    """Run the silver build pipeline for resolved kinds, grouping schema versions when configured."""
    bronze_table = bronze_table or globals().get("bronze_table")
    if not bronze_table:
        raise ValueError("bronze_table is not configured")
    timings: dict[str, float] = {}
    t0 = perf_counter()
    bronze_df = None
    bronze_cached = False
    run_info_rows: list[tuple] = []
    manifest_rows: list[tuple] = []
    output_docs_rows: list[tuple] = []
    run_id = str(uuid.uuid4())
    results: list[KindResult] = []
    output_mode = "wide" if reassemble else "normalized"
    write_mode = "upsert" if incremental else "full_refresh"
    merge_keys = _effective_merge_key_columns(merge_key_columns)
    watermark_column = globals().get("watermark_column") or ""
    watermark_mode = globals().get("watermark_mode", "auto")
    metadata_batching = bool(globals().get("batch_metadata_writes", True))
    flush_interval = max(1, int(globals().get("metadata_flush_interval", 100)))

    try:
        t_stage = perf_counter()
        schema_access_detail = validate_adme_schema_service_access()
        timings["schema_access_check"] = perf_counter() - t_stage

        t_stage = perf_counter()
        bronze_df, bronze_cached = prepare_bronze_df(
            spark,
            workspace_id,
            lakehouse_id,
            bronze_table,
            cache_enabled=bool(globals().get("cache_bronze", True)),
            storage_level_name=globals().get("bronze_cache_storage_level", "MEMORY_AND_DISK"),
        )
        timings["bronze_load_cache"] = perf_counter() - t_stage

        t_stage = perf_counter()
        kinds = ensure_resolved_kinds(kinds, spark, workspace_id, lakehouse_id, bronze_table, bronze_df=bronze_df)
        groups = group_kinds_by_version_strategy(kinds, version_strategy)
        timings["kind_resolution"] = perf_counter() - t_stage

        watermark_state = load_incremental_watermark_state(spark, kinds, watermark_column) if _watermark_active(incremental, watermark_column, watermark_mode) else {}

        kind_counts: dict[str, int] | None = None
        if globals().get("preflight_kind_counts", True) and not _watermark_active(incremental, watermark_column, watermark_mode):
            try:
                t_stage = perf_counter()
                kind_counts = compute_kind_counts(bronze_df, kinds)
                timings["kind_count_preflight"] = perf_counter() - t_stage
            except Exception as exc:
                logger.warning("Kind count preflight failed; falling back to per-kind counts: %s", exc)
                kind_counts = None

        schema_registry = None
        schema_status: dict[str, str] = {}
        if globals().get("schema_preflight", True):
            t_stage = perf_counter()
            schema_registry, schema_status = prefetch_schema_registry(kinds, enabled=True)
            timings["schema_preflight"] = perf_counter() - t_stage

        print(f"Silver build run: {run_id}")
        print(f"  Kinds: {len(kinds)}")
        print(f"  Version groups: {len(groups)}")
        print(f"  Version strategy: {version_strategy}")
        print(f"  Missing schema mode: {missing_schema_mode}")
        print(f"  Write mode: {write_mode}")
        print(f"  Output mode: {output_mode}")
        print(f"  Merge key columns: {merge_keys}")
        print(f"  Allow overwrite: {allow_overwrite}")
        print(f"  Include inactive records: {globals().get('include_inactive_records', False)}")
        print(f"  Bronze cached: {bronze_cached}")
        print(f"  Metadata batching: {metadata_batching} (flush interval {flush_interval})")
        print(f"  Watermark: {watermark_column or 'none'} ({watermark_mode})")
        print(f"  ADME schema access: {schema_access_detail}")
        if notebook_version:
            print(f"  Notebook version: {notebook_version}")
        if config_hash:
            print(f"  Config hash: {config_hash}")
        print(f"  Limit: {limit or 'none'}")
        print()

        for group_index, group in enumerate(groups, 1):
            group_kinds = list(group["kinds"])
            start_time = datetime.now(UTC)
            error_message = None
            error_type = None
            group_results: list[KindResult] = []
            t_group = perf_counter()
            try:
                group_results = process_kind_group(
                    spark,
                    group,
                    workspace_id,
                    lakehouse_id,
                    bronze_table=bronze_table,
                    limit=limit,
                    kind_limits=kind_limits,
                    incremental=incremental,
                    drop_wkt=drop_wkt,
                    reassemble=reassemble,
                    table_prefix=table_prefix,
                    allow_overwrite=allow_overwrite,
                    run_id=run_id,
                    notebook_version=notebook_version,
                    config_hash=config_hash,
                    missing_schema_mode=missing_schema_mode,
                    bronze_df=bronze_df,
                    kind_counts=kind_counts,
                    schema_registry=schema_registry,
                    schema_status=schema_status,
                    merge_key_columns=merge_keys,
                    output_docs_rows=output_docs_rows if metadata_batching else None,
                    watermark_column=watermark_column,
                    watermark_mode=watermark_mode,
                    watermark_state=watermark_state,
                )
            except Exception as exc:
                error_type = type(exc).__name__
                error_message = traceback.format_exc()
                logger.error("Failed to process version group %s: %s", group["group_key"], exc)
                parent_table = table_name_for_kind_group(group, table_prefix)
                group_results = [KindResult(kind=kind, status="failed", parent_table=parent_table, error=str(exc)) for kind in group_kinds]
            timings["group_processing"] = timings.get("group_processing", 0.0) + (perf_counter() - t_group)

            end_time = datetime.now(UTC)
            for result in group_results:
                results.append(result)
                result_error_type = error_type if result.status == "failed" else None
                info_row = run_info_row(
                    run_id,
                    result.kind,
                    start_time,
                    end_time,
                    result,
                    result.error or error_message,
                    result_error_type,
                    write_mode,
                    output_mode,
                    merge_keys,
                    schema_access_detail,
                    timings,
                    watermark_column or None,
                    watermark_mode,
                )
                manifest_row = run_manifest_row(
                    run_id,
                    result,
                    output_mode,
                    table_prefix,
                    bronze_table,
                    run_profile,
                    notebook_version,
                    config_hash,
                    allow_overwrite,
                    write_mode,
                    merge_keys,
                    watermark_column or None,
                    watermark_mode,
                )
                if metadata_batching:
                    run_info_rows.append(info_row)
                    manifest_rows.append(manifest_row)
                else:
                    run_info_rows.append(info_row)
                    manifest_rows.append(manifest_row)
                    flush_metadata_buffers(spark, run_info_rows, manifest_rows, workspace_id, lakehouse_id)

                status_icon = {"success": "✓", "skipped": "–", "schema_missing": "–", "failed": "✗"}.get(result.status, "?")
                print(f"  {status_icon} {result.kind}")
                print(f"    table: {result.parent_table}")
                print(f"    rows:  {result.records_processed}")
                if result.reassembled:
                    print("    output mode: wide")
                elif result.child_tables:
                    print("    output mode: normalized")
                    print(f"    children: {', '.join(result.child_tables)}")
                if result.error:
                    print(f"    error: {result.error}")
                print()

            if metadata_batching and len(run_info_rows) >= flush_interval:
                t_flush = perf_counter()
                try:
                    flush_metadata_buffers(spark, run_info_rows, manifest_rows, workspace_id, lakehouse_id)
                except Exception as exc:
                    logger.error("Metadata flush failed after group %d: %s", group_index, exc)
                timings["metadata_flush"] = timings.get("metadata_flush", 0.0) + (perf_counter() - t_flush)

                t_docs_flush = perf_counter()
                try:
                    flush_output_documentation_rows(spark, output_docs_rows, workspace_id, lakehouse_id)
                except Exception as exc:
                    logger.error("Output documentation flush failed after group %d: %s", group_index, exc)
                timings["output_docs_flush"] = timings.get("output_docs_flush", 0.0) + (perf_counter() - t_docs_flush)

    finally:
        if run_info_rows or manifest_rows:
            t_flush = perf_counter()
            try:
                flush_metadata_buffers(spark, run_info_rows, manifest_rows, workspace_id, lakehouse_id)
            except Exception as exc:
                logger.error("Final metadata flush failed: %s", exc)
            timings["metadata_flush"] = timings.get("metadata_flush", 0.0) + (perf_counter() - t_flush)
        if output_docs_rows:
            t_docs_flush = perf_counter()
            try:
                flush_output_documentation_rows(spark, output_docs_rows, workspace_id, lakehouse_id)
            except Exception as exc:
                logger.error("Final output documentation flush failed: %s", exc)
            timings["output_docs_flush"] = timings.get("output_docs_flush", 0.0) + (perf_counter() - t_docs_flush)
        if bronze_cached and bronze_df is not None:
            try:
                bronze_df.unpersist()
            except Exception as exc:
                logger.warning("Failed to unpersist bronze cache: %s", exc)

    timings["total"] = perf_counter() - t0
    succeeded = sum(1 for r in results if r.status == "success")
    failed = sum(1 for r in results if r.status == "failed")
    skipped = sum(1 for r in results if r.status in {"skipped", "schema_missing"})
    total_rows = sum(r.records_processed for r in results)
    print(f"Done: {succeeded} succeeded, {failed} failed, {skipped} skipped ({total_rows} total rows)")
    print("Timing summary:")
    for name, elapsed in sorted(timings.items(), key=lambda item: item[0]):
        print(f"  {name}: {elapsed:.1f}s")

    return results


## Setup checklist

Run this section before the smoke test or full pipeline when onboarding a new workspace. It validates the resolved tenant configuration, bronze access, ADME schema service access, and planned output tables without writing Silver Layer tables.


In [ ]:
# Setup checklist: validate configuration and planned outputs before writes
kind_selectors = globals().get("kind_selectors", kinds)
setup_checks = run_setup_checklist(
    spark,
    kinds=kinds,
    workspace_id=workspace_id,
    lakehouse_id=lakehouse_id,
    bronze_table=bronze_table,
    output_mode=output_mode,
    table_prefix=table_prefix,
    limit=limit,
    kind_limits=kind_limits,
    incremental=incremental,
    watermark_column=watermark_column,
    watermark_mode=watermark_mode,
)


## Smoke test bronze access

Run this section before full execution to validate that the configured bronze table and first selected kind can be read. The smoke test reads at most one row and does not write Silver Layer tables.


In [ ]:
# Smoke test: validate bronze access before running the full silver build
print("Smoke test starting...")
print(f"  bronze_table = {bronze_table}")
print(f"  include inactive records = {include_inactive_records}")

smoke_kinds = ensure_resolved_kinds(globals().get("kind_selectors", kinds), spark, workspace_id, lakehouse_id, bronze_table)
print(f"  resolved kinds = {len(smoke_kinds)}")
print(f"  sample kind    = {smoke_kinds[0] if smoke_kinds else '<none>'}")

if smoke_kinds:
    smoke_df = read_bronze_kind_spark(
        smoke_kinds[0],
        spark,
        workspace_id,
        lakehouse_id,
        bronze_table=bronze_table,
        limit=1,
    )
    sample_rows = smoke_df.take(1)
    print(f"  bronze preview rows = {len(sample_rows)}")
    print(f"  bronze preview cols = {len(smoke_df.columns)}")
else:
    print("  No kinds available for smoke test")


## Run pipeline

Run this section after reviewing the configuration output and completing the setup checklist and smoke test.

Profiles:

- `interactive`: print current settings and next steps without executing the pipeline.
- `dry_run`: validate selected kinds, resolve schemas, preview output tables, and avoid writes.
- `full`: process the configured kinds and write Silver Layer Delta tables.

Recommended order: configure controls, run Setup checklist, run Smoke test, set `RUN_PROFILE = "dry_run"`, run this section, then set `RUN_PROFILE = "full"` when ready.


In [ ]:
# ── Execute ───────────────────────────────────────────────────────────
if "decompose_kind" not in globals() or "reassemble_kind" not in globals() or "build_registry_from_adme" not in globals():
    print("Standalone core is not loaded yet.")
    print("Run the helper and core sections first, then rerun this cell.")
    results = []
elif run_profile == "interactive":
    print("Interactive profile active: no pipeline execution was started.")
    print("Current settings:")
    print(f"  kind selectors: {len(globals().get('kind_selectors', kinds))}")
    print(f"  schema source mode: {schema_source_mode}")
    print(f"  write mode: {write_mode}")
    print(f"  watermark: {watermark_column or 'none'} ({watermark_mode})")
    print(f"  include inactive records: {include_inactive_records}")
    print(f"  output mode: {output_mode}")
    print(f"  table prefix: {table_prefix}")
    print("Next steps:")
    print("  1. Update the explicit tenant configuration if needed.")
    print("  2. Run Configuration to refresh effective values.")
    print("  3. Run the setup checklist.")
    print("  4. Run the smoke test to validate bronze access.")
    print("  5. Set RUN_PROFILE to 'dry_run' for a write-free preview.")
    print("  6. Set RUN_PROFILE to 'full' when ready to write Silver Layer tables.")
    results = []
elif run_profile == "dry_run":
    dry_run_results = run_silver_dry_run(
        spark,
        kinds=globals().get("kind_selectors", kinds),
        workspace_id=workspace_id,
        lakehouse_id=lakehouse_id,
        bronze_table=bronze_table,
        limit=limit,
        output_mode=output_mode,
        table_prefix=table_prefix,
        kind_limits=kind_limits,
    )
    results = []
else:
    print("Full profile active: starting pipeline execution.")
    results = run_silver_build(
        spark,
        kinds=globals().get("kind_selectors", kinds),
        workspace_id=workspace_id,
        lakehouse_id=lakehouse_id,
        bronze_table=bronze_table,
        limit=limit,
        incremental=incremental,
        drop_wkt=drop_wkt,
        reassemble=reassemble,
        table_prefix=table_prefix,
        allow_overwrite=allow_overwrite,
        run_profile=run_profile,
        notebook_version=NOTEBOOK_VERSION,
        config_hash=config_hash,
        kind_limits=kind_limits,
        version_strategy=version_strategy,
        missing_schema_mode=missing_schema_mode,
        merge_key_columns=merge_key_columns,
    )


## Results summary

Review per-kind status, record counts, output table names, validation status, and errors after pipeline execution.


In [ ]:
# ── Results table ─────────────────────────────────────────────────────
import pandas as pd

rows = []
for r in results:
    rows.append(
        {
            "Kind": r.kind.split("--")[-1].split(":")[0],
            "Status": r.status,
            "Rows": r.records_processed,
            "Parent": r.parent_table,
            "Children": "wide" if r.reassembled else (len(r.child_tables) if r.child_tables else 0),
            "Validation": "✓" if r.validation_passed else "✗",
            "Error": r.error or "",
        }
    )

if rows:
    df_results = pd.DataFrame(rows)
    display(df_results)
elif "dry_run_results" in globals() and dry_run_results:
    display(pd.DataFrame(dry_run_results))
else:
    print("No pipeline results to display yet. Run with RUN_PROFILE = 'dry_run' or 'full'.")
